# 🧬 Analog Propagation — CASMI 2026 (V33: the AP-2 composition)

> **TL;DR.** The retrieval engine is unchanged: mass-shifted analog propagation over training structures ∪ COCONUT, four evidence channels and a calibrated gradient-boosted ranker. Three stages now act on the ranker's ordering: a **popularity prior**, a **forward isomer check** with a spectrum simulator, and a **gated PubChem-only tier at fixed slots**. Together they scored **0.340 twice** on the public leaderboard. The previous version of this notebook scored 0.330, and its card shows 0.335.
>
> **This version (V33) changes one number in the forward stage: GLACIER only (weight 1.0) instead of ICEBERG 0.5 + GLACIER 0.5.** Everything else is byte-identical to the composition that scored 0.340 twice. The change was not yet scored when this was written; its own result will appear on this notebook's card and in the ledger of the next version.

**How to read the scores.** The public leaderboard covers about a third of roughly 400 hidden molecules, so about 132 of them. Moving one molecule to rank 1 is worth about 0.0076. Our identical reruns have differed by up to 0.007, and two byte-identical public notebooks scored 0.005 apart. Treat differences below about 0.01 as noise. The ledger below lists **every** official submission we made, including the ones that went down.

**What runs.** One code cell, below the text. It needs a **T4 GPU** for the forward models and takes about an hour on the commit run. Everything of the previous public version (V32) is kept further down as inactive raw text, so nothing is lost and none of it executes.


## 1. The engine (unchanged since V32)

The metric is MRR@25 with an **exact** InChIKey-14 match, so we retrieve candidates rather than generate structures. For each molecule:

1. **Candidates.** Training structures are rebuilt at run time from `train.parquet` (competition data is never redistributed). COCONUT natural products are added. A ±10 ppm neutral-mass window gives a median of about 52 candidates and kept every validation answer.
2. **Evidence channels.**
   - *Library match*: entropy similarity against reference spectra, indexed on molecular-formula mass.
   - *Analog propagation*: spectra of structurally related molecules line up under a constant mass shift. A candidate scores `max_a sim(a)³ × Tanimoto(candidate, a)` over instrument-matched analogs within ±200 Da.
   - *Fragmentation*: MetFrag-lite (bond and bond-pair cleavage, ±2 H).
   - *Fingerprint model*: a merged-spectrum model and a single-spectrum model predict 6,930 fingerprint logits, and a candidate is scored by `f · z`. Pairing the two models was the largest single effect we measured: 0.330 vs 0.311.
3. **Ranker.** 31 features feed 8 HistGradientBoosting classifiers (2 class priors × 4 seeds), fitted in the notebook from the shipped feature rows. Their predictions are averaged.
4. **Stage 0.** One slot per answer the official scorer can tell apart (heavy-atom composition plus tautomer-canonical InChIKey-14).


## 2. Popularity prior

Most remaining errors are between isomers that share a formula. Among the top 60 pool candidates, we re-order by

$$z(\text{ranker probability}) + 0.20\,\big(\log(1+\#\text{PubChem substances}) + \log(1+\#\text{PubMed articles})\big)$$

Only pool candidates move, and only among the positions pool candidates already hold. A molecule whose best library match is ≥ 0.90 is left alone, because a near-identical reference spectrum beats any prior. The counts come from PubChem's CID–SID and CID–PMID files, keyed by InChIKey-14 (`pubchem-popularity-counts`).

**Measured:** S0 0.331 → **0.339** with the prior. It is not spectral evidence. It favours well-documented compounds and can push a famous isomer above the true one, which is why it is applied only inside the top 60, and only off-library.


## 3. PubChem-only tier, at fixed slots

The organisers define Class 2 as "in PubChem **or** COCONUT", but our pool is training structures ∪ COCONUT. So for molecules without a strong library match, we search a 71-million-structure PubChem mass index (natural-product formulas) in two passes:

1. Morgan-bit partial `f · z` over the whole ±10 ppm window. The top 5,000, plus the 200 most documented structures, go to pass 2.
2. Exact `f · z`, then the order `z(f · z) + 0.25 · popularity`. Up to 25 structures not already in the pool are kept.

These structures enter the final list **only at fixed positions**: 4, 8, 12, 16 and 20. They get 2, 4, 6, 8 and 10 when the tier's best `f · z` beats the pool's best by more than 300. They never pass through the ranker.

**Why fixed slots.**
- Merging PubChem isomers into the ranked pool scored **0.205** against 0.335 (September). The ranker's analog-Tanimoto features reward near-duplicates of the analogs, and a large isomer set supplies many of them.
- A matched diagnostic on 16 previously exposed public queries (Oct 7) let the rankers score the PubChem proposals together with the pool. In both queries where the tier was open, PubChem took 18–21 of the 25 positions, and one answer was lost (MRR 0.9417 fixed vs 0.9375 joint). Fixed slots stay.

**Measured:** tier alone 0.331 (no change). With the prior, **0.340**.


## 4. Forward isomer check: simulate the spectrum, re-order within a formula

98 % of wrong winners share the true formula, and analog Tanimoto cannot separate isomers. So inside each same-formula group of the top 60 (and of the PubChem tier), we re-sort by

$$z(\text{ranker}) + \lambda_I\,z(\text{ICEBERG}) + \lambda_G\,z(\text{GLACIER})$$

Each model term is the mean entropy similarity between the model's predicted spectrum and the measured spectra. Group members go back into the **same slots**, so this stage can only permute isomers. Library hits (≥ 0.90) are not touched.

- **Models.** ICEBERG and GLACIER from [ms-pred](https://github.com/coleygroup/ms-pred) (MIT), MassSpecGym checkpoints. The host has said MassSpecGym-trained models, and GLACIER by name, are allowed.
- **Packaging.** We packaged them ourselves: ms-pred at commit `708148c`, torch 2.6 + DGL in an isolated subprocess, RDKit 2025.03.6 for fragmentation. GLACIER is scored on **[M+H]+** spectra only. ICEBERG uses [M+H]+ and [M+Na]+.
- **Measured.** Forward alone: S0 0.331 → **0.338**. On top of popularity + tier (λ_I = λ_G = 0.5): 0.340 = no further gain.

**Why GLACIER only.** On the public two-engine notebooks, other authors ran single-variable comparisons. `lszlst`'s b420 (ICEBERG + GLACIER) scored 0.402 and b421 (GLACIER only) scored 0.417. `bobthebot369` reports +0.009 for isolated GLACIER. ICEBERG was trained on Orbitrap/QTOF spectra, while the test is 100 % timsTOF. **This version sets λ_I = 0 and λ_G = 1.0.** The total forward weight is unchanged, and ICEBERG is still computed, but it carries no weight. The rule was fixed before scoring: keep it if two draws average ≥ 0.348, drop it if they average ≤ 0.341.


## 5. How the code cell runs (and why it looks unusual)

The composition was developed and scored in a private research notebook. To publish **exactly the bytes that were scored**, the code cell below carries them as one sha256-pinned payload: Version 6's 21 code cells, with **one changed line** (`CFG.FWD_WEIGHTS = {'iceberg': 0.0, 'glacier': 1.0}`), plus a commit-only forward check. Then:

- **Real files.** Each payload cell is written to `/kaggle/working/f1_run/cells/<label>.py` and compiled with that real path. Numba's `@njit(cache=True)` needs a source file. Our first attempt without one failed in 12 s, so that is fixed and tested.
- **One owned process group.** The cells run in a fresh session, with a hard deadline: **7,130 s** on the commit run and **30,600 s** on the competition rerun, inside Kaggle's 9 h. On success and on timeout, every process in that group is stopped (TERM, then KILL), so no simulator or worker pool survives.
- **Pins.** The Version 6 code asserts the sha256 of every input file and of the fitted 8-model ranker, so a changed dataset or environment stops the run instead of silently scoring something else.
- **Checks before writing.** Columns, row order and ids are compared with `sample_submission.csv` and `test.parquet`. Each row must have 1–25 guesses, all parseable, none empty or repeated. On the commit run, the visible lists must equal Version 6's byte for byte (sha256 `0d1cad26…`; the visible test is a copy of training rows, so no gate opens), and the forward models must return finite scores on this GPU.
- **Output.** `/kaggle/working/submission.csv` is written **last**. Receipts, the payload log, the forward service logs and the composition diagnostics stay in `/kaggle/working/f1_run/`.


## 6. Results ledger: every official submission

All rows are our own submissions to this competition (public leaderboard). "S0" = the V32 engine with one slot per scorer-distinct answer. Rows marked *system error* failed on Kaggle's side and have no score.

| Date (UTC) | Submission | Public | What changed |
|---|---|---:|---|
| 2026-10-06 | 56883630 | — | V6 composition, exact redraw: Kaggle "system error" (no score) |
| 2026-10-06 | 56883185 | — | ICEBERG native-SUM reducer (V16), retry: Kaggle "system error" (no score) |
| 2026-10-06 | 56882879 | — | ICEBERG native-SUM reducer (V16), redraw: Kaggle "system error" (no score) |
| 2026-10-05 | 56861238 | 0.339 | V6 composition with ICEBERG's native SUM fragment reducer (V16) |
| 2026-10-05 | 56859903 | 0.340 | V6 composition, exact redraw |
| 2026-10-04 | 56824628 | 0.340 | **V6 composition**: popularity prior + PubChem tier (fixed slots) + forward check (ICEBERG 0.5 + GLACIER 0.5) |
| 2026-10-04 | 56820767 | 0.330 | S0 + per-ion 10 ppm precursor union |
| 2026-10-04 | 56818694 | 0.338 | S0 + forward check only (ICEBERG + GLACIER) |
| 2026-10-04 | 56818650 | 0.331 | S0, matched control in the current container |
| 2026-10-03 | 56801583 | 0.339 | S0 + popularity prior |
| 2026-10-03 | 56801550 | 0.331 | S0 + PubChem tier (fixed slots) |
| 2026-10-03 | 56801511 | 0.340 | S0 + popularity prior + PubChem tier |
| 2026-10-03 | 56797952 | 0.331 | S0 = V32 engine + one slot per scorer-distinct answer (replicate) |
| 2026-10-03 | 56797936 | 0.331 | S0 = V32 engine + one slot per scorer-distinct answer |
| 2026-09-21 | 56443438 | 0.330 | v17 = this notebook's V32 recipe: formula-mass library index, instrument-matched analogs, N_ANALOG 200 |
| 2026-09-21 | 56440933 | 0.328 | v16: exact replication of the 0.335 configuration |
| 2026-09-21 | 56439294 | 0.330 | v15: v14 + N_ANALOG 200 |
| 2026-09-21 | 56436020 | 0.327 | v14: library indexed on molecular-formula mass |
| 2026-09-19 | 56368289 | 0.311 | v13: later merged checkpoint (m1@36k) in the model pair |
| 2026-09-19 | 56364259 | 0.321 | v12: PubChem off, ranker depth 10 |
| 2026-09-19 | 56359610 | 0.205 | v11: PubChem isomers merged into the ranked pool |
| 2026-09-18 | 56335354 | 0.329 | v10: a third fingerprint model |
| 2026-09-18 | 56327285 | 0.311 | v9: merged-only model channel (m1@36k) |
| 2026-09-18 | 56325700 | 0.330 | v8: back to the m1@24k + s2 model pair |
| 2026-09-17 | 56303325 | 0.311 | v7: no candidate cap, 10 ppm window, 50k-step model |
| 2026-09-17 | 56301107 | 0.282 | v6: 50k-step merged model, cap 80, 8.5 ppm window |
| 2026-09-17 | 56297964 | 0.335 | v5: merged + single model pair, seed-averaged ranker (this notebook's card score) |
| 2026-09-16 | 56287238 | 0.292 | v4: single model s2, W1 0.50 (second draw) |
| 2026-09-16 | 56287236 | 0.298 | v4: single model s2, W1 0.50 |
| 2026-09-16 | 56284092 | 0.295 | v3: four channels, model ensemble |
| 2026-09-16 | 56284090 | 0.299 | v3: four channels (library, analogs, fragmentation, fingerprint model) |
| 2026-09-16 | 56274988 | 0.266 | v2: + MetFrag-lite fragmentation |
| 2026-09-15 | 56261625 | 0.245 | v1 public baseline: analog propagation + calibrated ranker |
| 2026-09-15 | 56259296 | 0.233 | entropy library + mass-shifted analog propagation, GBM ranker |
| 2026-09-15 | 56257940 | 0.151 | library-only probe (measures the Class-1 share) |

Best: **0.340** (rows 56824628 and 56859903, the composition in this notebook). Largest single effects: the model pair (+0.019) and analog propagation itself (0.151 → 0.245). Largest loss: PubChem merged into the ranked pool (0.205).


In [ ]:
# =====================================================================================================
#  The only code cell of this notebook. It runs the AP-2 composition described in the text above:
#  analog-propagation engine -> popularity prior -> forward isomer check -> gated PubChem tier (fixed slots).
#  This version: forward weights ICEBERG 0.0 / GLACIER 1.0 (GLACIER scores [M+H]+ only); all else = V6.
#
#  The composition's code (21 cells of our private research worker's Version 6, plus a commit-only forward
#  check) travels below as one sha256-pinned zlib/base64 payload, so the bytes that run are the bytes that
#  were scored. Each payload cell is written to /kaggle/working/f1_run/cells/<label>.py and executed in one
#  process group of its own (a fresh session), with a hard deadline: 7,130 s on the commit run, 30,600 s on
#  the competition rerun. The whole group is stopped (TERM, then KILL) on success and on timeout. The
#  submission is checked against sample_submission.csv and test.parquet (ids, order, 1-25 parseable unique
#  guesses), and /kaggle/working/submission.csv is written as the very last step. On the commit run the
#  visible lists must equal Version 6's (sha256 0d1cad26...) and the forward models must answer on this GPU.
#  Every cell of the previous public version (V32) is kept below as inactive raw text or unchanged markdown.
# =====================================================================================================
import time as _f1_time
F1_T0 = _f1_time.monotonic()
import base64, glob, hashlib, json, os, pathlib, shutil, signal, subprocess, sys, threading, zlib

F1_CONFIG = dict(input_root='/kaggle/input', work_root='/kaggle/working', commit_cap_s=7130, rerun_cap_s=30600, reserve_s=150, require_gpu=True, require_canary=True, payload_override=None, placeholder_sig='654e030e4173780e945252b4a5adf70b', v6_visible_csv_prefix='0d1cad26', term_grace_s=30, kill_grace_s=15)

F1_PAYLOAD_B64 = '...'
F1_PAYLOAD_ZLIB_BYTES = 49480
F1_PAYLOAD_ZLIB_SHA256 = '8398ef14250240706b60b751fa7c28d094182f1f10b740c0210a381887a6aca7'
F1_PAYLOAD_JSON_BYTES = 141493
F1_PAYLOAD_JSON_SHA256 = '5cc3bf3e8e42e4400633a8d40ca1f73da9fb3dc3607ab2d509e2b9242e7e7053'

F1_RUNNER = r'''
import hashlib, json, os, re, sys, time
def _f1_run(path):
    # F1b: every payload cell is materialised as a real file and compiled with that path, so code that needs
    # a source file (numba @njit(cache=True), inspect.getsource, tracebacks) behaves as it did in the kernel.
    payload = json.load(open(path, encoding='utf-8'))
    g = sys.modules['__main__'].__dict__
    cells_dir = os.path.join(os.getcwd(), 'cells')  # cwd = this run's own fresh f1_run directory
    os.makedirs(cells_dir, exist_ok=False)
    for cell in payload['cells']:
        src, label = cell['source'], cell['label']
        if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*', label):
            raise RuntimeError('F1 payload label not a safe file name: ' + label)
        if hashlib.sha256(src.encode('utf-8')).hexdigest() != cell['sha256']:
            raise RuntimeError('F1 payload cell drift: ' + label)
        cell_path = os.path.join(cells_dir, label + '.py')
        with open(cell_path, 'x', encoding='utf-8', newline='') as f:
            f.write(src)
        with open(cell_path, 'rb') as f:
            if hashlib.sha256(f.read()).hexdigest() != cell['sha256']:
                raise RuntimeError('F1 materialised cell differs: ' + label)
        t = time.monotonic()
        print('F1_CELL_START', label, flush=True)
        exec(compile(src, cell_path, 'exec'), g)
        print('F1_CELL_DONE', label, round(time.monotonic() - t, 1), flush=True)
    print('F1_PAYLOAD_COMPLETE', flush=True)
_f1_run(sys.argv[1])
'''

class F1Hold(RuntimeError):
    pass


def _f1_require(ok, message):
    if not ok:
        raise F1Hold('F1 HOLD: ' + message)


def _f1_sha(data):
    return hashlib.sha256(data).hexdigest()


def _f1_find(root, name):
    hits = glob.glob(os.path.join(root, '**', name), recursive=True)
    _f1_require(bool(hits), 'missing input ' + name)
    return sorted(hits, key=len)[0]


def _f1_mode(cfg):
    import pandas as pd
    test = _f1_find(cfg['input_root'], 'test.parquet')
    ids = pd.read_parquet(test, columns=['spectrum_id']).spectrum_id.astype(str)
    signature = hashlib.md5(','.join(sorted(ids)).encode()).hexdigest()
    env_flag = bool(os.environ.get('KAGGLE_IS_COMPETITION_RERUN'))
    return dict(is_rerun=bool(env_flag or signature != cfg['placeholder_sig']), env_flag=env_flag,
                test_signature=signature, comp_dir=os.path.dirname(test), spectra=int(len(ids)))


def _f1_gpu():
    exe = shutil.which('nvidia-smi')
    if not exe:
        return dict(ok=False, reason='nvidia-smi not found')
    try:
        r = subprocess.run([exe, '-L'], capture_output=True, text=True, timeout=60)
    except Exception as e:  # noqa: BLE001 - recorded, then gated
        return dict(ok=False, reason=repr(e)[:300])
    gpus = [line.strip() for line in r.stdout.splitlines() if line.startswith('GPU ')]
    return dict(ok=bool(r.returncode == 0 and gpus), gpus=gpus, returncode=r.returncode)


def _f1_payload():
    raw = base64.b64decode(F1_PAYLOAD_B64, validate=True)
    _f1_require(len(raw) == F1_PAYLOAD_ZLIB_BYTES and _f1_sha(raw) == F1_PAYLOAD_ZLIB_SHA256,
                'payload compressed binding')
    decoder = zlib.decompressobj()
    text = decoder.decompress(raw, F1_PAYLOAD_JSON_BYTES + 1)
    _f1_require(len(text) == F1_PAYLOAD_JSON_BYTES and decoder.eof and not decoder.unconsumed_tail
                and _f1_sha(text) == F1_PAYLOAD_JSON_SHA256, 'payload decoded binding')
    payload = json.loads(text.decode('utf-8'))
    for cell in payload['cells']:
        _f1_require(_f1_sha(cell['source'].encode('utf-8')) == cell['sha256'], 'payload cell ' + cell['label'])
    return text, payload


def _f1_parse_stat(stat):
    """(state, pgrp) from a Linux /proc/<pid>/stat line; comm may contain spaces and parentheses."""
    rest = stat[stat.rfind(')') + 2:].split()
    return rest[0], int(rest[2])


def _f1_proc_table():
    """[(pid, state, pgrp)] for every visible process (Linux /proc; `ps` elsewhere, e.g. local tests)."""
    table = []
    if os.path.isdir('/proc'):
        for entry in os.listdir('/proc'):
            if not entry.isdigit():
                continue
            try:
                with open('/proc/%s/stat' % entry) as f:
                    state, pgrp = _f1_parse_stat(f.read())
            except (OSError, ValueError, IndexError):
                continue
            table.append((int(entry), state, pgrp))
    else:
        r = subprocess.run(['ps', '-A', '-o', 'pid=,pgid=,stat='], capture_output=True, text=True)
        for line in r.stdout.splitlines():
            p = line.split()
            if len(p) >= 3 and p[0].isdigit() and p[1].isdigit():
                table.append((int(p[0]), p[2][:1], int(p[1])))
    return table


def _f1_group_members(pgid):
    """Live (non-zombie) processes whose process group is pgid."""
    return [pid for pid, state, pgrp in _f1_proc_table() if pgrp == pgid and state not in ('Z', 'X')]


def _f1_leader_exited(pid):
    """True once the (unreaped) leader is a zombie or gone; it is never reaped here."""
    for p, state, _ in _f1_proc_table():
        if p == pid:
            return state in ('Z', 'X')
    return True


def _f1_run_owned(cmd, cwd, env, deadline, log_path, cfg):
    """Run cmd as the leader of a new session; the leader stays unreaped until its group is empty."""
    log = open(log_path, 'ab', buffering=0)
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, start_new_session=True)
    pgid = os.getpgid(proc.pid)

    def pump():
        for line in iter(proc.stdout.readline, b''):
            log.write(line)
            sys.stdout.write(line.decode('utf-8', 'replace'))
            sys.stdout.flush()

    reader = threading.Thread(target=pump, daemon=True)
    reader.start()
    timed_out = False
    while True:
        if _f1_leader_exited(proc.pid):
            break
        if _f1_time.monotonic() >= deadline:
            timed_out = True
            break
        _f1_time.sleep(2)
    signals_sent = []
    for sig, grace in ((signal.SIGTERM, cfg['term_grace_s']), (signal.SIGKILL, cfg['kill_grace_s'])):
        live = _f1_group_members(pgid)
        if not live:
            break
        try:
            os.killpg(pgid, sig)
        except ProcessLookupError:
            break
        signals_sent.append(dict(signal=sig.name, live_members=len(live)))
        end = _f1_time.monotonic() + grace
        while _f1_time.monotonic() < end and _f1_group_members(pgid):
            _f1_time.sleep(0.5)
    remaining = _f1_group_members(pgid)
    returncode = proc.wait(timeout=60)  # reap the leader only after its group is empty
    reader.join(timeout=15)
    log.close()
    return dict(pid=proc.pid, pgid=pgid, own_group=pgid == proc.pid, returncode=returncode,
                timed_out=timed_out, signals=signals_sent, remaining_members=remaining,
                cleanup_proven=bool(pgid == proc.pid and not remaining))


def _f1_validate(csv_path, mode, cfg):
    import pandas as pd
    raw = pathlib.Path(csv_path).read_bytes()
    sub = pd.read_csv(csv_path)
    samp = pd.read_csv(os.path.join(mode['comp_dir'], 'sample_submission.csv'))
    test_ids = set(pd.read_parquet(os.path.join(mode['comp_dir'], 'test.parquet'),
                                   columns=['molecule_id']).molecule_id.astype(str))
    ids = sub.molecule_id.astype(str) if 'molecule_id' in sub.columns else pd.Series([], dtype=str)
    smiles = sub.smiles if 'smiles' in sub.columns else pd.Series([], dtype=object)
    rows = [str(s).split(';') for s in smiles.fillna('')]
    checks = dict(
        columns=list(sub.columns) == ['molecule_id', 'smiles'],
        rows_equal_sample=len(sub) == len(samp),
        order_equals_sample=list(ids) == list(samp.molecule_id.astype(str)),
        ids_equal_test=set(ids) == test_ids,
        no_duplicate_ids=not ids.duplicated().any(),
        no_null_smiles=not smiles.isnull().any() and len(smiles) == len(sub),
        guesses_1_to_25=all(1 <= len(r) <= 25 for r in rows),
        no_empty_guess=all(all(g.strip() for g in r) for r in rows),
        no_repeated_guess_in_row=all(len(set(r)) == len(r) for r in rows))
    try:
        from rdkit import Chem, RDLogger
        RDLogger.DisableLog('rdApp.*')
        bad = sum(1 for r in rows for g in r if Chem.MolFromSmiles(g) is None)
        checks['all_guesses_parse'] = bad == 0
    except ImportError:
        checks['all_guesses_parse'] = 'skipped'
    digest = _f1_sha(raw)
    if not mode['is_rerun']:
        checks['visible_csv_equals_v6'] = digest.startswith(cfg['v6_visible_csv_prefix'])
    return checks, digest, dict(rows=len(sub), guesses=sum(len(r) for r in rows))


def _f1_write_receipt(run_dir, receipt):
    receipt['elapsed_s'] = round(_f1_time.monotonic() - F1_T0, 1)
    (run_dir / 'F1_RUN_RECEIPT.json').write_text(json.dumps(receipt, indent=2, default=str))
    print('F1_RUN_RECEIPT', json.dumps({k: receipt.get(k) for k in
          ('status', 'elapsed_s', 'cap_s', 'submission_sha256', 'checks', 'error')}, default=str), flush=True)


def F1_MAIN(cfg):
    work = pathlib.Path(cfg['work_root'])
    run_dir = work / 'f1_run'
    run_dir.mkdir(parents=True, exist_ok=False)
    receipt = dict(arm='F1_v6_glacier_only', runner='file-backed cells, owned process group', change="V6 cell 3: CFG.FWD_WEIGHTS {'iceberg': 0.0, 'glacier': 1.0}",
                   status='STARTED', config=cfg, official_score=None)
    try:
        mode = _f1_mode(cfg)
        receipt['mode'] = mode
        cap = cfg['rerun_cap_s'] if mode['is_rerun'] else cfg['commit_cap_s']
        deadline = F1_T0 + cap - cfg['reserve_s']
        receipt['cap_s'] = cap
        receipt['gpu'] = _f1_gpu()
        if cfg['require_gpu']:
            _f1_require(receipt['gpu']['ok'], 'GPU required: the forward channel only runs on CUDA')
        if cfg['payload_override']:
            payload_path = pathlib.Path(cfg['payload_override'])
            receipt['payload'] = dict(override=str(payload_path), sha256=_f1_sha(payload_path.read_bytes()))
        else:
            text, payload = _f1_payload()
            payload_path = run_dir / 'f1_payload.json'
            payload_path.write_bytes(text)
            receipt['payload'] = dict(sha256=F1_PAYLOAD_JSON_SHA256, cells=[c['label'] for c in payload['cells']],
                                      v6_source_sha256=payload.get('v6_source_sha256'))
        runner = run_dir / 'f1_runner.py'
        runner.write_text(F1_RUNNER, encoding='utf-8')
        env = dict(os.environ, MPLBACKEND='Agg', PYTHONUNBUFFERED='1',
                   F1_IS_RERUN='1' if mode['is_rerun'] else '0')
        cmd = [sys.executable, '-u', str(runner), str(payload_path)]
        process = _f1_run_owned(cmd, str(run_dir), env, deadline, run_dir / 'payload.log', cfg)
        receipt['process'] = process
        _f1_require(not process['timed_out'], 'payload exceeded the %d s cap' % cap)
        _f1_require(process['returncode'] == 0, 'payload returncode %s' % process['returncode'])
        _f1_require(process['cleanup_proven'], 'owned process group not empty after teardown')
        csv_path = run_dir / 'submission.csv'
        _f1_require(csv_path.is_file(), 'payload wrote no submission.csv')
        checks, digest, size = _f1_validate(csv_path, mode, cfg)
        receipt.update(checks=checks, submission_sha256=digest, submission_size=size)
        failed = sorted(k for k, v in checks.items() if v is not True and v != 'skipped')
        _f1_require(not failed, 'submission checks failed: %s' % failed)
        canary_path = run_dir / 'F1_FORWARD_CANARY.json'
        receipt['forward_canary'] = json.loads(canary_path.read_text()) if canary_path.is_file() else None
        if cfg['require_canary'] and not mode['is_rerun']:
            _f1_require(bool(receipt['forward_canary']) and receipt['forward_canary'].get('status') == 'PASS',
                        'forward canary did not pass on this machine')
        receipt['status'] = 'PASS_SUBMISSION_WRITTEN_LAST'
        _f1_write_receipt(run_dir, receipt)
        tmp = work / 'submission.csv.f1tmp'
        shutil.copyfile(csv_path, tmp)
        os.replace(tmp, work / 'submission.csv')  # the last write of this notebook
        _f1_require(_f1_sha((work / 'submission.csv').read_bytes()) == digest, 'copied submission differs')
        return receipt
    except BaseException as error:
        receipt['status'] = 'HOLD'
        receipt['error'] = repr(error)[:3000]
        _f1_write_receipt(run_dir, receipt)
        raise


F1_RECEIPT = F1_MAIN(F1_CONFIG)


## 7. Credits, data and licences

**Attached inputs**

| Input | What it is | Licence / terms |
|---|---|---|
| Competition data (`enveda-CASMI26-molecule-id-mass-spectra`) | spectra, training structures, test | **CC BY-NC 4.0 (non-commercial)**. Competition rules forbid redistribution, which is why the training half of the pool is rebuilt at run time |
| [`prvsiyan/casmi26-fp-models-v2`](https://www.kaggle.com/datasets/prvsiyan/casmi26-fp-models-v2) | the two fingerprint checkpoints (ours) | labelled CC0, but **trained on the competition's CC BY-NC training data**: treat as non-commercial-derived |
| [`prvsiyan/casmi26-ranker-features`](https://www.kaggle.com/datasets/prvsiyan/casmi26-ranker-features) | 142,762 × 31 ranker training rows (ours) | labelled CC0, **derived from competition data** (same caveat) |
| [`prvsiyan/coconut-casmi26-candidates`](https://www.kaggle.com/datasets/prvsiyan/coconut-casmi26-candidates) | 436,389 COCONUT structures, masses, fingerprints | **CC BY 4.0**: please cite COCONUT ([coconut.naturalproducts.net](https://coconut.naturalproducts.net/)) |
| [`prvsiyan/pubchem-npformula-massindex`](https://www.kaggle.com/datasets/prvsiyan/pubchem-npformula-massindex) | 71 M-structure PubChem mass index (ours) | CC0 label; source: NCBI PubChem public downloads |
| [`prvsiyan/pubchem-popularity-counts`](https://www.kaggle.com/datasets/prvsiyan/pubchem-popularity-counts) | substance / PubMed counts per InChIKey-14 (ours) | CC0 label; source: PubChem CID–SID / CID–PMID files |
| [`prvsiyan/rdkit-wheel-offline`](https://www.kaggle.com/datasets/prvsiyan/rdkit-wheel-offline) | RDKit 2026.03.3 wheel (the scorer's version) | RDKit: BSD-3-Clause |
| [`prvsiyan/casmi26-forward-reference-stack`](https://www.kaggle.com/datasets/prvsiyan/casmi26-forward-reference-stack) | ms-pred source (MIT), ICEBERG / GLACIER MassSpecGym checkpoints (MIT), torch 2.6 / DGL / PyG wheels, RDKit 2025.03.6 | **Other** (mixed): MIT code and weights. Third-party wheels keep their own licences, including NVIDIA's CUDA EULA for the CUDA runtime wheels. If this input is unavailable to you, the run stops at the forward check rather than silently skipping it |

The previously attached `chebi-lipidmaps-casmi26` (ChEBI + LIPID MAPS, **CC BY-NC-SA 4.0**) is no longer attached. It was switched off in V32 and lost 0.004 when it was on.

**Ideas we credit (no code copied):**
- The popularity prior, the PubChem-only list with fixed slots and the library gate follow the design of [Ahmed Berat Özer](https://www.kaggle.com/ahmedberatozer) and the public fusion notebooks of `wangpenghua`, `seyitkaangunes`, `huseyinemreaksoy` and `dmitriigluzdov`.
- GLACIER-only weighting was reported by `lszlst` and `bobthebot369`.
- ICEBERG and GLACIER are by the Coley group (ms-pred). MassSpecGym is by Bushuiev et al.
- The two-model insight, the analog-propagation engine and the honest-noise discussion below are ours.

**Licence of this notebook's code:** Apache 2.0 (Kaggle's default for public notebooks).


---
## Previous public version (V32), preserved and inactive

Everything below is V32 of this notebook, in its original order.
- The 20 **markdown** cells are unchanged. They are the full write-up of the engine, the measurements behind each setting, and the noise-floor discussion.
- The 17 **code** cells are kept as **raw** cells with their source byte-for-byte, so they never execute.

The V32 recipe itself scored 0.330 (row 56443438). This notebook's card score of 0.335 came from the earlier v5 recipe (row 56297964).


# 🧬 Analog Propagation — a retrieval baseline for CASMI 2026

> **TL;DR** — MRR@25 here is an *exact-structure* metric, so generating SMILES is the wrong tool.
> This notebook **retrieves** from a 711,705-structure library and spends all its effort on
> *ordering* that list. The centrepiece is **mass-shifted analog propagation**, which identifies
> molecules that have **no reference spectrum anywhere** by matching them against their chemical
> relatives.

| Stage | What it adds | Public LB |
|---|---|---|
| Spectral library search only | finds Class-1 molecules | **0.151** |
| \+ analog propagation & calibrated ranker | reaches Class 2 | **0.233** |
| \+ leaderboard-calibrated class weighting | stops over-paying for Class 2 | **0.245** |
| \+ in-silico fragmentation channel | independent structural evidence | **0.266** |
| \+ spectrum→fingerprint model | chemistry predicted from the spectrum | **0.299** |
| \+ merged-spectrum model, seed-averaged ranker | matching train to test | **0.335** |
| \+ constants copied from a popular fork | **a regression** — one of them was a recall bug | **0.282** |
| \+ removing the candidate cap, 10 ppm window | restores 24% of Class-2 recall | **0.311** |
| \+ reverting every constant to its highest-scoring value | local validation preferred all of them; the leaderboard did not | **0.330** |
| \+ isolating the model channel (single-variable A/B) | m1@36k alone vs m1@24k+s2: **−0.019** | **0.311** |
| \+ a third model in the ensemble | ensembling *within* a view does not help | **0.329** |
| \+ PubChem pool expansion, top-50 isomers by `f·z` | bought the answer to $\rho$; the pool already had the answers | **0.205** |
| \+ PubChem off, tree depth 10 (query-held-out CV) | even the *honest* CV does not transfer for ranker capacity | **0.321** |
| \+ back to the exact configuration that scored 0.335 | after eight experiments, nothing beat it | **0.335** |
| \+ formula-mass library index, instrument-matched analogs | two mechanical defects fixed | **0.327** |
| \+ `N_ANALOG` re-swept to 200 on the fixed channel | | **0.330** |
| **re-run of the 0.335 row, changing nothing** | **it scored 0.328 — see below** | **0.328** |

**Read this table with its noise band.** The last row is the *same code* as the 0.335 row. The gap
between them is 0.007, which is exactly the seed-noise range measured elsewhere in this notebook.
So 0.335 was the top of the band rather than a level, the family mean is **0.329 ± 0.004**, and
most of the apparent regressions above are differences between a configuration and itself. The one
effect that clears the noise is the **model pair**: 0.327–0.335 against 0.311/0.311/0.311 for
merged-only. See *Running the same thing twice*.

The two regression rows are left in on purpose; they are the most useful numbers in the table.

**0.282** — five constants were adopted from a well-scoring fork **as a bundle**. Four were fine;
one silently deleted a quarter of the reachable answers. The calibrated objective predicted the
drop at −0.057 before the leaderboard reported −0.053. See *The candidate cap*.

**0.311** — that fixed the cap but was still below 0.335, because three more changes had been
bundled in alongside it. Rather than guess, I spent a submission on a proper single-variable A/B:
two runs identical in every respect **except** the model channel. `m1@36k` merged-only scored
**0.311**; `m1@24k + s2` scored **0.330**. So the model channel carried the whole regression, the
analog constants were innocent, and — the useful part — **averaging a per-spectrum model with a
merged model is worth +0.019**, which is the largest single effect I have measured on this
leaderboard. Both local holdouts said the opposite. See *When validation stops predicting the
leaderboard*.

Four independent channels, one calibrated ranker. Every number in this notebook is measured — the
dead ends are reported alongside the wins, because knowing that PubChem *hurts* is worth as much as
knowing that analog propagation helps.

Everything is configurable from one `CFG` block. Each constant below is annotated with the
measurement that chose it, so you can see what is load-bearing and what is not.

---

## Why retrieval, and not generation

A prediction scores only if its **InChIKey14 matches exactly** after tautomer canonicalisation.
Getting a ~350 Da natural product exactly right by autoregressive decoding is vanishingly unlikely —
one misplaced hydroxyl and the score is zero. Ranking a finite list of *real* molecules turns an
impossible generation problem into a tractable ordering problem.

## The test set has three very different halves

The hosts split the ~400 scored molecules by novelty, and **hide which is which**:

| Class | Definition | The only thing that can find it |
|---|---|---|
| **1** | has public reference MS/MS | spectral library search |
| **2** | in PubChem/COCONUT, **no** reference spectra | database retrieval + ranking |
| **3** | not in PubChem at all | de novo |

The mix is hidden — so I measured it. A library-only submission of this engine scores **0.151**,
and the same engine scores **0.93 MRR** on a Class-1 simulation. That puts Class 1 at roughly
**0.151 / 0.93 ≈ 16%** of the test set.

**So ~84% of the test is out of reach of library search**, and that is where nearly all the
remaining score lives. Most public baselines spend all 25 slots on library hits ranked by cosine —
which is why the leaderboard clusters just above the Class-1 ceiling.


---

## 🔑 The main idea: mass-shifted analog propagation

A Class-2 molecule has **no reference spectrum**. But its **structural relatives usually do**.

Two molecules that share a scaffold and differ by one substituent fragment into *largely the same
ions, offset by a constant mass*. So a **modified (mass-shifted) similarity** still matches a
spectrum against relatives of molecules the library has never seen. Propagating those relatives'
known structures onto candidates gives:

$$\mathrm{score}(c)\;=\;\max_{a\,\in\,\text{analogs}}\;\mathrm{sim}(a)^{p}\cdot\mathrm{Tanimoto}(f_c,\,f_a)$$

**Measured on a held-out Class-2 simulation (250 natural products, all their spectra removed from
the library):**

| Ranking of the same candidates | MRR@25 |
|---|---|
| random order | 0.138 |
| by mass error *(what public baselines do)* | 0.164 |
| NP-likeness prior | 0.037 |
| **analog propagation** | **0.521** |

It needs no Class-1 branch either: for a Class-1 molecule the library contains the molecule
*itself*, which shows up as an analog at zero mass shift with Tanimoto 1.0 and is ranked first
automatically.

It also generalises. Repeating the experiment on **569 obscure** held-out structures (rather than
common natural products) gives **0.516** — statistically identical, so this is not an artefact of
well-studied compounds.


---

## The candidate pool, and why it is *small on purpose*

Candidates = **training-set structures ∪ COCONUT**, deduplicated on InChIKey14 →
**711,705** structures with precomputed fingerprints. COCONUT covers **99.6%** of
`enveda-np-examples`, the library the hosts describe as closest to the test set.

A ±10 ppm neutral-mass window gives a **median of 56 candidates**, and contained the true structure
**100%** of the time in validation (timsTOF precursor error stays under ~9 ppm, with a systematic
+1.4 ppm calibration offset).

**Bigger is worse — this is measured, not assumed:**

| Candidate window | median candidates | Class-2 MRR |
|---|---|---|
| **±10 ppm** | **56** | **0.521** |
| ±20 ppm | 76 | 0.509 |
| ±30 ppm | 105 | 0.500 |
| \+ all PubChem isomers (~3,900 more) | ~4,000 | **0.350** |
| \+ only the *top 10* PubChem isomers | 66 | **0.337** |

Adding PubChem is catastrophic, and note that admitting only its **top 10** is just as bad. The
reason is instructive: analog-Tanimoto is maximised by candidates that are **near-duplicates of the
analogs**, whereas the true molecule is usually a *derivative* (median Tanimoto to its best analog
is 0.83, most often ±CH₂ or ±O away). COCONUT works precisely *because* it does not contain those
near-duplicate decoys; any large isomer set supplies them in bulk and order statistics guarantee
several out-rank the truth.

### Does the fingerprint model rescue PubChem? No. (I checked.)

The obvious hypothesis is that PubChem only failed because analog-Tanimoto is the wrong scorer for
it, and that a model scoring each structure **intrinsically** would fix it. I tested exactly that
with the full four-channel ranker:

| candidate pool | Class-2 MRR |
|---|---|
| COCONUT + train only | **0.732** |
| \+ top-25 PubChem | 0.379 |
| \+ top-100 PubChem | 0.328 |
| \+ top-500 PubChem | 0.338 |

Still catastrophic. The arithmetic settles it: adding PubChem is net-positive only when

$$g \;>\; \frac{0.35\,\rho}{1-\rho}$$

where $\rho$ is your current pool recall and $g$ is the MRR you achieve on the molecules your pool
misses. At a plausible $\rho \approx 0.6$ that demands $g > 0.52$ — better than this ranker manages
on a *56*-candidate list, let alone 4,000. **Do not spend your time here.**

### What does work: targeted expansion

ChEBI + LIPID MAPS adds only **+8.8%** candidates but **+7–19%** coverage of the structures in the
public spectral libraries (ChEBI brings metabolites, LIPID MAPS brings lipids — both regions a
plant/microbe-focused NP database under-covers). Dilution cost is **−0.026** with the model channel
(it was −0.035 without), which the coverage gain should more than repay. Toggle with
`CFG.USE_BIO_DB`.

> **Open problem for you:** ~57% of truths sit within one or two common biosynthetic deltas
> (+CH₂, +O, +hexose, …) of their best analog. Expanding the pool with *targeted derivatives*
> should raise recall — but only once the ranker can tell **which** derivative. Note that
> positional isomers give *different* fragment-mass sets, so the in-silico fragmentation channel
> is the one with a chance of telling them apart.


### Building the other half of the pool, at runtime

The attached dataset contains **only the COCONUT half** of the candidate pool. The competition
rules forbid redistributing competition data to non-participants, and the training structures come
from `train.parquet` — so the notebook rebuilds that half itself, from the file you already have.

It costs ~5 minutes of RDKit on the Kaggle CPU and makes the pipeline fully reproducible: nothing
about the candidate pool is hidden inside a precomputed blob you cannot inspect.

The fingerprint is `ECFP4(4096) ‖ ECFP6(4096) ‖ RDKitFP(2048) ‖ MACCS(167)`, reduced to the
**6,930** bits whose frequency across training structures lies in [0.5%, 99.5%] (`fp_bits.npy`).
Rare bits carry no discrimination and common bits carry no information.


### How the candidate pool was built (so you can rebuild or extend it)

The attached `casmi26-pool` dataset is derived entirely from the competition data plus one public
database, and is reproducible in ~10 minutes:

1. **COCONUT 2.0** (`coconut_csv-09-2026.zip`, CC-BY, <https://coconut.naturalproducts.net/download>)
   — parse `canonical_smiles`, keep the largest fragment (drops salts/counterions), reject charged
   species and anything outside 100–1300 Da or 5–120 heavy atoms → **462,028** unique InChIKey14.
2. **Training-set structures** from `train.parquet` → **275,810** unique InChIKey14.
3. Union, deduplicated on **InChIKey14** (the metric is stereochemistry-blind, so two spellings of
   one skeleton would only waste a slot) → **711,705**, sorted by exact mass.
4. Fingerprint per structure: `ECFP4(4096) ‖ ECFP6(4096) ‖ RDKitFP(2048) ‖ MACCS(167)`, then keep
   only bits whose frequency across the training structures lies in **[0.5%, 99.5%]** →
   **6,930 bits**, bit-packed. Shipping them precomputed is why this notebook needs **no RDKit**.

Swapping in your own database is a drop-in change: produce `pool_mass.npy`, `pool_fp.npy`
(bit-packed, same 6,930 bits) and `pool_meta.pkl` with `keys`/`smiles`, and everything downstream
works unchanged.

### Credits

* **Entropy similarity** — Li, Kind, Fiehn et al., *Nature Methods* 2021.
* **COCONUT 2.0** — Chandrasekhar, Steinbeck et al., *Nucleic Acids Research* 2025.
* **CSI:FingerID / fingerprint retrieval** — Dührkop, Böcker et al., *PNAS* 2015 (the `f·z`
  identity in the last section is the linear-algebra shortcut through their scoring function).
* Analog / mass-shifted matching is the standard **GNPS molecular-networking** idea, applied here
  as a *ranking prior over a candidate database* rather than for network visualisation.


---

## 🧪 Third channel: in-silico fragmentation (MetFrag-lite)

Analog propagation borrows structure from a *neighbour*. This channel instead judges each candidate
**on its own merits**: break its bonds, see whether the resulting fragments can explain the peaks
that were actually observed.

For each candidate we break every single bond, and every *pair* of bonds, keep the connected
components, allow ±2 hydrogen rearrangements, and score the fraction of (square-rooted) peak
intensity that lands within `MZ_TOL` of some fragment ion.

| Channel | Class-2 MRR |
|---|---|
| in-silico fragmentation alone | 0.259 |
| analog propagation alone | 0.521 |
| **both (fixed blend)** | **0.545** |

The two are genuinely independent — their correlation on the *true* structures is only **0.058** —
which is exactly why combining them helps. It costs ~14 ms per candidate, so a few minutes for the
whole test set.

It is also the one channel that is **not** fooled by near-duplicates of an analog, which is why it
was worth testing whether it rescues a PubChem-sized candidate pool. It does not (see above) — but
it is a clean, training-free source of evidence, and a natural place to plug in a real fragmenter
(CFM-ID, SIRIUS) if you want to go further.


---

## 🧠 Channel 4: spectrum → fingerprint, and why ranking is *literally* a dot product

The first three channels all reason by **comparison** — to a library spectrum, to an analog, to a
candidate's own bond-breaking. This one predicts chemistry **directly from the spectrum**: a
transformer reads the peaks and outputs a 6,930-bit molecular fingerprint.

To rank a candidate with fingerprint $f$ under predicted per-bit logits $z$, the natural score is
the Bayes log-likelihood of its bits:

$$\sum_i \big[f_i\log\sigma(z_i) + (1-f_i)\log\sigma(-z_i)\big]$$

Now use the identity $\log\sigma(z)-\log\sigma(-z) = z$ (exactly, for all $z$). Splitting the sum:

$$= \sum_i f_i\big[\log\sigma(z_i)-\log\sigma(-z_i)\big] + \sum_i\log\sigma(-z_i)
  \;=\; \boxed{f\cdot z} \;+\; \underbrace{\textstyle\sum_i\log\sigma(-z_i)}_{\text{identical for every candidate}}$$

**Ranking by the full Bayesian score is a plain dot product with the raw logits.** No sigmoid, no
calibration, no temperature. Two consequences:

1. Inference is one matrix multiply.
2. The ranking objective can be trained **end-to-end**: each training spectrum is scored against
   63 decoy structures sampled from the *same ±10 ppm mass window* — precisely the competitors it
   will face at inference — under a softmax cross-entropy on $f\cdot z$.

### What it is worth

| Channel (alone) | Class-2 MRR@25 |
|---|---|
| in-silico fragmentation | 0.259 |
| **fingerprint model** | **0.468** |
| analog propagation | 0.521 |
| analog + model | **0.567** |
| all four, via the calibrated ranker | **0.612** |

The model is nearly as strong as analog propagation *by itself*, and because it reasons from
different evidence the two combine rather than overlap.

### Training notes (the mistake worth avoiding)

With only ~276k distinct structures, this model **memorises fast**. My first run looked healthy and
then quietly rotted:

| step | 9k | 12k | 40k | 69k |
|---|---|---|---|---|
| held-out hard-negative top-1 | 0.446 | **0.457** | ~0.30 | **0.127** |

Training loss kept improving the whole time. If you save the *last* checkpoint — as I did — you
ship a model three times worse than the one you had at step 12k. Two fixes are in the training
script: **keep the best-by-validation checkpoint**, and **augment** (peak dropout, intensity jitter,
±5 ppm m/z noise), which pushed the peak to 0.467 and moved it out to ~20k steps.


---

## 🔬 Two fixes from the community that the four channels were quietly paying for

Both of these came out of other people's public work, and both are the kind of defect that
validation *can* catch: mechanical, not a matter of taste. The version carrying them scored 0.327,
then 0.330 once the analog count was re-swept, against a same-code replication of the previous
configuration at 0.328 — neutral-to-slightly-positive on the leaderboard and clearly right on the
mechanics. They ship ON, behind `CFG.LIB_MASS_FORMULA` and `CFG.ANALOG_MATCH_INSTR` so you can A/B
them yourself.

### 1. A ppm window is the wrong instrument for an absolute error

[@dariushafshar](https://www.kaggle.com/code/dariushafshar/casmi26-your-ppm-window-drops-121-805-spectra)
derived every training spectrum's neutral mass twice — once from `molecular_formula`, which is
arithmetic, and once from `precursor_mz + adduct`, which is what every published pipeline here
(including this one) actually indexes on — and found **121,805 spectra disagree by more than
±10 ppm**. The errors have two shapes:

* **reported precision.** `riken` is accurate to about 0.005 Da. That is a fine *absolute* error
  and a terrible *relative* one: 0.005 Da is 17 ppm at 300 Da and 5 ppm at 1000 Da, so a ppm
  window fails at the light end — exactly where this competition lives.
* **wrong adduct labels.** `gnps` misses cluster at discrete offsets: −1.008 Da is one hydrogen,
  +21.982 Da is sodium mistaken for a proton. Those spectra are indexed at the wrong mass entirely.

The suggested fix is a hybrid tolerance, `max(10 ppm, 0.01 Da)`. This notebook does something
stricter: **index on the formula mass directly.** It is exact arithmetic, so it is immune to both
error modes at once, and it makes the index *tighter* instead of looser:

| library index | Class-1 MRR | library spectra inspected/query |
|---|---|---|
| precursor + adduct, 10 ppm | 0.9206 | 571 |
| precursor + adduct, max(10 ppm, 0.01 Da) | 0.9232 | 914 |
| **formula mass, 10 ppm** | **0.9253** | **663** |

Better *and* narrower. It falls back to the precursor derivation where the formula is missing or
contains an untabulated element (0.0002% of spectra).

### 2. Analog propagation was comparing across instruments

This one is mine, and it had been sitting in `build_rep` since the first version. The rule was
"one representative spectrum per structure, pick the richest." But analog propagation compares a
**shifted neighbour spectrum peak by peak** — the neighbour's instrument decides which fragments
are present at all and at what relative intensity. Richest-wins was routinely handing a timsTOF
query an Orbitrap neighbour.

The test set is **100% timsTOF**, and the library holds 1.15M timsTOF spectra (45%), so a
same-instrument representative usually exists:

| representative choice | Class-2 analog MRR |
|---|---|
| richest spectrum (old) | 0.5241 |
| prefer any TOF/QTOF | 0.5311 |
| **prefer the exact instrument** | **0.5412** |

The exact match is worth **more than double** the loose one — `QTOF` is not `timsTOF`, and treating
them as interchangeable throws away most of the gain. Together with the formula index the analog
channel goes **0.5213 → 0.5412**, a 3.8% relative gain on the largest Class-2 lever.

Note *why* this one is trustworthy when so many local measurements here are not: the holdout is
timsTOF natural products and the test is timsTOF molecules, so on the instrument axis specifically
the domain gap that defeats everything else is closed. The preferred instrument is read off the
test set at runtime rather than hardcoded, so it adapts if the hidden set is mixed.

**Why it is the instrument specifically.** Three sibling ideas, same harness, all measured:

| representative rule (inside the ±200 Da analog window) | analog c₂ |
|---|---|
| **prefer the test's instrument**, then richest | **0.5510** |
| prefer instrument, then match ionization mode | 0.5386 |
| prefer instrument, then 2 spectra per structure | 0.5296 |
| prefer instrument, then closest to the test's median collision energy | 0.4966 |

Only the instrument helps, and the failures explain why. A query contributes **several** spectra —
at 20, 40 and 60 eV, in one polarity — and the analog score takes the **max** over them, so
collision energy and ionization mode are already covered by the query side. Spending the
representative slot on matching them throws away spectrum richness for information you had anyway;
CE matching costs 0.054. The instrument is different because it is fixed for the whole
experiment: no query spectrum is an Orbitrap spectrum, so nothing on the query side can compensate.

Keeping two spectra per structure is still worse (0.5296), which confirms the original finding —
extra representatives raise the max similarity of irrelevant structures faster than of the right
one, and instrument-matching does not rescue it.

### …and the parameter it unlocked

`N_ANALOG` was one of this notebook's documented dead ends: 60/80/100/140 analogs scored
0.3683/0.3709/0.3705/0.3703 predicted LB — flat, nothing to win. Re-swept on the instrument-matched
channel it is flat no longer:

| analogs kept | 80 | 200 | 600 | 800 |
|---|---|---|---|---|
| analog-channel c₂ | 0.5412 | 0.5510 | 0.5536 | **0.5595** |

The mechanism is the point. Before, analog #150 was as likely to be an Orbitrap spectrum as a
useful neighbour, so widening the list added noise as fast as signal. With the representatives
matched to the instrument, the extra analogs are real neighbours and keeping more pays. **A
parameter's optimum moves when the channel beneath it changes** — the same lesson as the class
prior moving after the model channel improved, and the reason a "measured dead end" deserves
re-measuring after any structural fix.

Shipped at 200 rather than 800: the surface past 200 is noisy (300 and 450 both dip below it) and
250 queries is not enough to chase a 0.008 tail.

Altogether the analog channel goes **0.5213 → 0.5510** — formula index, instrument match and the
re-swept parameter, none of which required a new model or a bigger database.

### 3. A metadata alignment bug, found by a reader

[@denpugovkin](https://www.kaggle.com/code/denpugovkin/casmi26-v17-adduct-grouped-merged-fpnet)
spotted that `_logits_from` dropped empty preprocessed spectra from the peak lists and then read
metadata from `rows[:B]` — pairing a surviving spectrum with a *different* spectrum's precursor,
adduct, instrument and collision energy. It fires only when `prep_peaks` rejects a spectrum
(0 of 1,213 on the visible test), but when it fires it corrupts the model channel for that
molecule. Fixed: the surviving spectra and their own metadata rows now travel together.


---

## ⚠️ The candidate cap: a recall bug that hides from validation

Several popular forks of this problem shrink the candidate list before ranking, keeping the best
`80` by

```python
coarse = library_sim * 100.0 - abs(pool_mass[cand] - target)
```

It reads like a sharper version of "tighter windows win" — prefer the library hits, then the
closest in mass. It is worth stepping through why it is not.

**A Class-2 molecule has `library_sim = 0` by definition.** That is what Class 2 *means*: the
structure exists in a database but no public reference spectrum does. So the `library_sim * 100`
term is identically zero for exactly the molecules the whole notebook is trying to identify, and
their ordering collapses to `-|Δmass|` — mass proximity *inside a window that was already selected
by mass*. At ±8.5 ppm around 500 Da the entire window spans 0.0085 Da; ranking candidates by where
they fall inside it is close to ranking them at random.

Truth retention on the Class-2 holdout, measured directly (does the correct structure survive the
cap at all?):

| cap | Class-1 retention | Class-2 retention |
|---|---|---|
| none | 0.992 | **0.992** |
| 400, by mass | 0.992 | **0.992** |
| 200, by mass | 0.992 | 0.952 |
| 80, by model `f·z` | 0.992 | 0.900 |
| **80, by mass** | 0.992 | **0.752** |

A cap of 80 discards **a quarter of the answers that were reachable**, before the ranker ever sees
them. No amount of ranking quality recovers them — MRR is bounded above by retention.

**Why this survives validation.** Look at the Class-1 column: it is 0.992 at *every* setting,
because `library_sim * 100` dominates and library hits are always kept. If you sanity-check a cap
on the channel that is easy to check, it looks free. The damage is confined to the class that has
no library evidence to protect it — and that class is ~⅔ of the scoring weight
(`B ≈ 0.271` vs `A ≈ 0.162`).

Running the full four-channel ranker either way, on the same seeds and the same features:

| | Class-1 MRR | Class-2 MRR | predicted LB |
|---|---|---|---|
| no cap | 0.922 | **0.789** | **0.363** |
| cap 80 by model `f·z` | 0.889 | 0.696 | 0.333 |
| cap 80 by mass | 0.896 | 0.603 | 0.308 |

**The arithmetic closes.** If retention were the *only* damage, capped MRR should be
uncapped MRR × retention ratio:

* cap 80 by mass: `0.789 × (0.752/0.992) = 0.598` — measured **0.603**
* cap 80 by `f·z`: `0.789 × (0.900/0.992) = 0.716` — measured **0.696**

So essentially the entire loss is answers that never reached the ranker, plus a small residual.
That residual has a cause too: `rank_features` contains **set-relative** features (`_rank_norm`
divides by `len(cand)-1`, `_z` subtracts the set mean). The ranker was fitted on *uncapped*
candidate sets, so capping at inference silently rescales those features. Removing the cap fixes
a train/inference mismatch as well as the recall loss.

The cap here is kept only as a **runtime guard** on in-silico fragmentation (~14 ms/candidate), set
to `500` — the median window holds 52 candidates and the largest observed holds 401, so it
never fired once across the 248 validation queries (at 80 it fired on 97 of them, 39%). When it
does fire it ranks by the model's `f·z`, which is actual evidence, rather than by mass, which is
not.

**This was not a hypothetical.** Adopting that fork's constants as a bundle took this notebook
from **0.335 → 0.282** on the public leaderboard. Decomposed against the calibrated objective:

| change | predicted ΔLB | source |
|---|---|---|
| candidate cap 80 by mass | **−0.055** | retention 0.992 → 0.752 |
| window 10 ppm → 8.5 ppm | −0.010 | retention 1.000 → 0.992 |
| 36k-step → 50k-step merged model | +0.008 | Class-2 channel 0.490 → 0.517 |
| **net** | **−0.057** | **observed −0.053** |

Four of the five constants were harmless (`N_ANALOG` 80→100 moves predicted LB by 0.0004; `SIM_POWER`
3→4 is a genuine small win). Only one was a bug — and it travelled attached to the reputation of the
others. **Adopt constants one at a time, and make each one show you its own measurement.**

The general lesson is worth more than the fix: **any pruning heuristic keyed on a feature that is
structurally absent for one class will silently delete that class.** Check retention per class, not
aggregate MRR.


---

## Combining the evidence: calibrate, don't hand-weight

Library similarity and analog evidence are on incomparable scales. Adding them with fixed weights
fails badly — putting library similarity in with weight 1.0 drops Class-2 MRR from **0.52 → 0.27**,
because for a Class-2 molecule *every* library hit is a wrong same-mass isomer.

Instead a small gradient-boosted model maps the evidence to a calibrated
**P(candidate is the answer)**, trained on a Class-1 simulation and a Class-2 simulation mixed at
the measured base rate.

### ⚠️ Two leaks to avoid if you build your own holdout

Masking a structure's spectra out of the library leaks in two ways, and both produced large fake
gains before being caught:

1. **Pool provenance.** Masking removes *evidence*, not *membership* — the answer is still a
   training-library structure in the pool. Feeding the ranker `src` (train vs COCONUT-only) or
   `np_likeness` (NaN for exactly the training structures) scored **0.94** on a task that honestly
   scores 0.55.
2. **Query in library.** A Class-1 simulation must drop the query's own *source library*, not just
   its InChIKey — otherwise it retrieves the identical spectrum and reports a perfect 1.000.

### Choosing the class weight

`CFG.W1` is **not** the Class-1 share. A Class-2 molecule is only worth something if the pool
actually contains it, so its value is discounted by pool recall. Solving the two leaderboard
readings —

* library-only `0.151` with Class-1 MRR `0.93` ⟹ Class-1 value `0.162`
* full ranker `0.233` with Class-1 MRR `0.73` ⟹ Class-2 value `0.22`

— gives `W1 = 0.162 / (0.162 + 0.22) ≈ 0.42`, roughly **2.6× more weight on Class 1** than the raw
share suggests. Sweeping it against that objective:

| `W1` | Class-1 MRR | Class-2 MRR | predicted LB |
|---|---|---|---|
| 0.16 *(raw share)* | 0.735 | 0.545 | 0.239 |
| **0.42** | **0.873** | **0.511** | **0.254** |
| 0.85 | 0.910 | 0.460 | 0.249 |

If your pool recall differs from mine, re-solve for `W1` — it is the single most leveraged constant
in the notebook.


---

## 🪤 When validation stops predicting the leaderboard — and how far you can actually diagnose it

This one cost a leaderboard place, and the honest version of the story is more useful than the
tidy one I first wrote.

**The observation.** Two model channels, measured on both local holdouts and on the leaderboard:

| model channel | 250-NP holdout | 569-obscure holdout | **public LB (whole config)** |
|---|---|---|---|
| `m1@24k` merged + `s2` per-spectrum, averaged | 0.4904 | 0.5579 | **0.335** |
| `m1@36k` merged alone | **0.5173** | **0.5624** | **0.311** |

Both local sets prefer the later checkpoint. The leaderboard preferred the configuration built on
the earlier pair, by 0.024. So the local numbers did not transfer.

**Those two scores came from configurations differing in four ways**, not one: the model channel,
`N_ANALOG` 80→100, `SIM_POWER` 3→4, and a newly added candidate cap. That is the candidate-cap
mistake again — several changes at once, then reasoning as if there had been one. So rather than
guess, I spent a submission on a proper **single-variable A/B**: two runs identical in every
respect except which model dataset was attached.

| configuration | public LB |
|---|---|
| `m1@24k` merged + `s2` per-spectrum | **0.330** |
| `m1@36k` merged alone | **0.311** |

**+0.019 for the two-model channel**, isolated. That also clears the analog constants: they were
innocent, and the model channel carried the entire regression. It is the largest single effect I
have measured on this leaderboard — and both local holdouts rank it the wrong way round.

**A mechanism I proposed, tested, and had to drop.** The obvious explanation is checkpoint
selection: the trainer saves on `if va > best_val`, so later checkpoints are the ones that won
more comparisons against a held-out split built from the same natural-product libraries my
holdouts come from. Selection pressure pointing at my own measuring stick.

It is a good story and the arithmetic kills it. The early-stopping check draws a **fresh random
2048-spectrum sample from 109,985 validation spectra** each time, so any given spectrum appears in
about **1.9%** of checks — roughly once across the whole run — and the decision it feeds is a
single aggregate scalar, about fifty times total. That is nowhere near enough selection pressure
to move a specific 250-structure subset by 0.027. I checked the ordinary leak too: all 250
validation structures sit in the model's held-out half, none in its training half.

So: real effect, mechanism unproven.

**What survives, and it is the part worth keeping.** Both holdouts are built the same way — mask a
structure's spectra out of a library that is overwhelmingly natural-product and timsTOF-heavy — and
the hidden test's Class-2 molecules are by definition ones with *no* reference spectra anywhere.
Those are different distributions, and a local MRR difference of 0.02–0.03 between two model
channels simply does not survive the trip. Note that the two local sets do not even agree with each
other on how big the gap is (0.027 vs 0.005), which is itself a warning that neither is measuring
what the leaderboard measures.

**What this notebook does about it**

1. **Ships the configuration that actually scored**, not the one that validates best. Every
   constant here is at the value from the highest-scoring submission, and where local validation
   disagrees, the local number loses.
2. **Keeps the two channels distinguishable.** Local validation is still used for things it *can*
   measure — candidate **retention** is a set-membership fact, not a model opinion, and the
   candidate-cap section above is built entirely on it. Prefer measurements that do not depend on
   a model's taste.
3. **Changes one thing per submission.** Both regressions in the results table came from bundling.

If you fork this and can afford the GPU time, the clean fix is a **three-way split**: train, a
model-selection split, and an evaluation split from a *different* library or instrument family
than either. Then a local number means something again.


---

## ⚠️ Before you trust any leaderboard delta: measure your noise floor

I submitted the **same notebook version twice** and got **0.292** and **0.298**.

Same code, same data, same everything. The culprit is one unset argument:
`HistGradientBoostingClassifier` defaults to `random_state=None`, so every fit bins and splits
differently. Reproducing it locally over five seeds:

| seed | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| predicted LB | 0.3033 | 0.3002 | 0.2961 | 0.3003 | 0.2977 |

**Range 0.0072** — which matches the 0.006 I saw on the actual leaderboard.

This is worth pausing on, because in a competition where the whole public leaderboard spans about
0.15–0.34, a **0.005 difference is indistinguishable from your ranker's seed**. I had to retract one
of my own conclusions because of it: I reported that adding ChEBI + LIPID MAPS *hurt*, based on
0.299 → 0.295. That gap sits inside the noise band. It is not evidence, and I should not have stated
it as firmly as I did.

**Why the two-model channel wins, and what that predicts.** The two models are not redundant —
they are fed *different input representations*. `s2` was trained one spectrum at a time and is fed
one spectrum at a time; `m1` was trained on fused multi-spectrum inputs (`--merge_p 0.6`) and is fed
a single merged peak list. Feeding either the other's view costs about 0.02, which is why
`load_model` routes by filename. So the ensemble is not "two draws of the same estimator" — it is
two genuinely different readings of the same spectra.

That distinction makes a testable prediction: adding models *within* a view should give ordinary,
much smaller ensembling gains, while having **both** views is the large, structural win. Worth
knowing before you spend GPU hours training a third model of the same kind.

**What this notebook does about it**

1. **Pin the seed** (`CFG.SEEDS`) so a rerun is reproducible.
2. **Average over seeds** — variance falls as ~1/√n *and* the mean improves slightly, because
   averaging decorrelated trees is just bagging.
3. **Average over class priors too** (`CFG.W1_PRIORS`) — and here is the mistake I made, left in
   because it is the most repeatable mistake in this whole competition.

   After the model channel improved I re-swept `W1` and tree depth, and got a beautiful result: a
   clean unimodal curve peaking at `W1 = 0.45`, a narrow plateau `(.40,.45,.50)` beating the shipped
   `(.30,.60)` by 0.008, and depth 6 → 8 → 10 adding another 0.007, each step improving **both**
   classes rather than trading one for the other. I checked it against the seed-noise band. I shipped
   it. It scored **0.330 against the 0.335 it replaced.**

   The sweep trained on all 819 query groups and then evaluated on 250 of them. **Those 250 were
   training rows.** Deeper trees and finer priors fit the evaluation queries better because the
   ranker had already seen them — I was measuring capacity to memorise my own test. The "both
   classes improve together" signal I trusted as evidence against noise is exactly what in-sample
   overfitting produces, since both classes share the same query groups.

   `make_ranker3.py` had always done this correctly, with 5 folds **held out by query group**
   (`folds[G] != f`). My quick sweep dropped the fold logic to go faster, and I did not notice that
   this changed what the number meant. The shipped values are the cross-validated ones.

   **The two rankings, side by side.** Same configs, same features, same seeds — the only
   difference is whether the evaluation queries were also training rows:

   | ranker config | in-sample sweep | **5-fold, held out by query** | actual LB |
   |---|---|---|---|
   | `(.30,.60)` depth 6 | 0.3477 | 0.3024 | **0.335 / 0.328** |
   | `(.30,.60)` depth 10 | 0.3479 | **0.3042** | — |
   | `(.45)` depth 6 | — | 0.3023 | — |
   | `(.40,.45,.50)` depth 6 | 0.3530 | 0.3003 | — |
   | `(.40,.45,.50)` depth 10 | **0.3603** | 0.3017 | **0.330** |

   The in-sample column ranks the configurations **almost exactly backwards**. Its top choice is
   the honest column's second-worst, and the two rows I actually submitted confirm the honest
   column's ordering, not the sweep's. In-sample numbers are also inflated by ~15% in absolute
   terms, but the inflation is not the problem — the *re-ordering* is.

   There is a real finding buried in there, and it only became visible once the evaluation was
   honest: **depth 10 is genuinely better, but only with the original priors** (0.3024 → 0.3042).
   The narrow plateau was the harmful half of what I bundled, and it was harmful enough to cancel
   the depth gain and then some. Two changes, opposite signs, shipped together — which is the same
   mistake as the candidate cap, made again, by me, after I had written the section warning about it.

   **Postscript: the honest CV did not transfer either.** Having caught the in-sample mistake, I
   shipped the config that the *correct* 5-fold query-held-out CV preferred — depth 10 with the
   original priors, worth +0.0018 out-of-sample. It scored **0.321**, against a same-code
   replication of the baseline at **0.328** — about 1.7 standard deviations down, and the only
   change in this notebook that still looks genuinely negative once the noise band was measured
   properly (see *Running the same thing twice*). So:

   | | in-sample sweep | query-held-out CV | actual LB |
   |---|---|---|---|
   | `(.30,.60)` depth 6 | 0.3477 | 0.3024 | **0.335 / 0.328** |
   | `(.30,.60)` depth 10 | 0.3479 | **0.3042** | **0.321** |

   Fixing the leak fixed the *ordering* of the prior configs — but it did not make the CV able to
   rank tree depth. The reason is the same domain gap that defeats every other local measurement
   here: the 250 held-out queries are natural products from the training libraries, and the hidden
   test is timsTOF molecules with no reference spectra. Extra capacity fits the local feature
   distribution and does not generalise across that gap.

   The practical rule this leaves: **on this competition, treat every local number as a filter, not
   a decision.** It can tell you something is broken (retention, leaks, recall) but it cannot tell
   you which of two working configurations is better. Those cost a submission each.

   **If you take one thing from this notebook, take this:**   **If you take one thing from this notebook, take this:** when you tune a ranker, hold out by
   *query*, not by row. Every candidate of one molecule must be on the same side of the split.
   Otherwise the model sees the answer's neighbours at training time and the number you get back is
   not a prediction of anything.

**The local noise band.** Fitting the same config with four *disjoint* seed sets (0-3, 4-7, 8-11,
12-15) at W1 = 0.45 gives 0.3811 / 0.3752 / 0.3784 / 0.3762 — a spread of **0.0059** around a mean
of 0.3777. That is the bar: an improvement smaller than ~0.006 on a single 4-seed run is not
distinguishable from the seed. The W1 curve above clears it because it is a *smooth shape across
seven points*, not one lucky measurement — which is the cheapest way to tell signal from noise
without spending submissions.

**What you should do about it:** before you accept that some change bought you +0.004, submit the
*unchanged* notebook twice and see how far apart the two scores land. It costs two submissions and
it will save you from optimising noise for a week.


---

## 🧪 The most useful submission I made: running the same thing twice

Every leaderboard number in this notebook had been treated as a *level* — 0.335 was the
configuration to beat, and anything scoring 0.330 or 0.327 was a regression to be explained. Three
days of work were organised around that reading.

So I spent a submission re-running the **exact configuration that scored 0.335**, changing nothing.

**It scored 0.328.**

| submission | configuration | LB |
|---|---|---|
| v5 | the four-channel pipeline, model pair | **0.335** |
| v8 | + narrow priors, depth 10 | 0.330 |
| v10 | + a third model | 0.329 |
| v12 | + depth 10, original priors | 0.321 |
| v14 | + formula index, instrument-matched analogs | 0.327 |
| v15 | + `N_ANALOG` 200 | 0.330 |
| **v16** | **identical to v5** | **0.328** |

v5 and v16 are the same code. The gap between them is **0.007** — the seed-noise range measured
independently much earlier in this notebook (0.0072, from submitting one unchanged version twice).
The whole family has mean 0.329 and sd 0.004.

**0.335 was the top of the noise band, not a level.** Which means:

* Most of the "regressions" in the table above never happened. v8, v10, v14 and v15 are all within
  one standard deviation of v16, i.e. of themselves.
* The formula index, the instrument-matched analogs and the re-swept `N_ANALOG` are **not**
  harmful. v15 (0.330) sits *above* the replication (0.328). They are shipped ON, because they are
  mechanically justified and they cost nothing.
* Depth 10 at 0.321 is the one change that still looks genuinely negative, and even that is only
  ~1.7 standard deviations.
* The single effect in this whole table that clears the noise decisively is the **model pair**:
  three runs at 0.327–0.335 against three merged-only runs at **0.311, 0.311, 0.311**.

**What this cost, and what it is worth.** I spent roughly two days explaining differences that were
noise, and writing careful post-mortems for regressions that did not exist. One submission —
the cheapest experiment available at any point — would have prevented all of it.

The rule I would give anyone forking this: **before you interpret a leaderboard delta, submit the
same notebook twice and measure your own noise band.** Then refuse to explain anything smaller than
it. The notebook already contained that advice, in the seed-noise section, written after two
identical versions scored 0.292 and 0.298. I wrote it and then spent two days not following it.


---

## 🔁 Appendix: how the fingerprint model was trained (so you can beat it)

The weights ship in an attached dataset, but a baseline you cannot retrain is a dead end. The full
training script is below — it is **not executed here** (it needs a GPU and ~2 h); copy it into a
separate notebook with a T4 and the two attached datasets.

**The objective.** Each training spectrum is scored against **63 decoy structures drawn from the
same ±10 ppm mass window** — exactly the competitors it meets at inference — under a softmax
cross-entropy on `f·z`. Because `f·z` *is* the Bayes log-likelihood (see the model section), this
trains the ranking function directly rather than a proxy.

**Three things that mattered, all learned the hard way:**

1. **Save the best-by-validation checkpoint, not the last.** With ~276k distinct structures the
   model memorises fast. Held-out hard-negative top-1 peaks around step 12–20k and then *decays*:

   | step | 9k | 12k | 40k | 69k |
   |---|---|---|---|---|
   | held-out top-1 | 0.446 | **0.457** | ~0.30 | **0.127** |

   Training loss improves the whole way down. My first run shipped the step-69k checkpoint.

2. **Augment.** Peak dropout, intensity jitter and ±5 ppm m/z noise moved the peak from 0.457 to
   0.467 and pushed it out to ~20k steps.

3. **Merge spectra during training — and then feed it merged spectra.** The test gives each molecule
   **1–16 spectra (median 3)**, but the obvious setup trains on single spectra — a real train/test
   mismatch. With `--merge_p 0.6`, 60% of training inputs are 2–4 spectra of the same structure
   collapsed into one peak list (near-duplicate m/z merged, stronger peak kept).

   The catch: **a model only wins on the input distribution it was trained on.**

   | | fed per-spectrum | fed one merged list |
   |---|---|---|
   | trained on single spectra (`fp_single_*`) | **0.4799** | 0.4597 |
   | trained on merged (`fp_merged_*`) | 0.4687 | **0.4896** |

   Feed either one the wrong way and you give up ~0.02 Class-2 MRR. Using each correctly and
   averaging the two gives **0.4904**. Beware also that a merge-trained model's *validation* number
   is computed on merged inputs and so is not comparable to a single-spectrum model's — its 0.511
   hard-negative top-1 looks far better than the single model's 0.453, but on the real Class-2 task
   the gap is much smaller. Always re-check on the task you actually care about.

**Numerical trap worth knowing.** `f·z` with ~800 on-bits and logits of ±5 overflows fp16 and the
softmax saturates. Compute the score in fp32 and centre it within each example —
`sc = (raw − raw.mean(dim=1)) / sqrt(nbits)` — which is a per-example constant shift, so the
ranking is untouched while the magnitudes stay sane. My first run diverged to NaN without it.

**Obvious things to try from here:** pretrain on the unlabelled GeMS/DreaMS corpora the hosts point
to, predict the molecular formula and condition on it, or replace the MetFrag-lite channel with a
real fragmentation-tree scorer (SIRIUS/CFM-ID).


---

## 🤖 A pretrained foundation model, and why it does not help *this* channel

The host has confirmed that open pretrained weights are allowed provided they are distributed in
compliance with the licence of the data they were trained on, so the obvious move is to reach for a
mass-spectrometry foundation model instead of the 20,000-step one trained here.

**[DreaMS](https://github.com/pluskal-lab/DreaMS)** is the right candidate on paper: MIT licence,
pretrained self-supervised on GeMS (GNPS public spectra — and `pluskal_ms2` is one of this
competition's own training libraries), 1024-dimensional embeddings, and explicitly designed so that
embedding similarity tracks *structural* similarity. Analog propagation is the strongest Class-2
channel here (isomer-benchmark MRR 0.541 against 0.169 random) and its similarity function is
hand-crafted entropy similarity, so replacing it with a learned one looks like free money.

It is not. Same queries, same analog windows, same propagation arithmetic — only `sim()` changes:

| analog similarity | n = 40 | **n = 130** |
|---|---|---|
| entropy on mass-shifted peaks (shipped) | 0.1700 | **0.1555** |
| DreaMS embedding cosine | 0.1604 | 0.1129 |
| entropy × √DreaMS | **0.1848** | 0.1428 |
| entropy × DreaMS^0.25 | — | 0.1526 |
| entropy × DreaMS^1 | — | 0.1409 |

(Absolute values are low because the analog windows are subsampled for this bench; only the
comparison matters.)

The first column is why the second exists. At 40 queries the product looked like a **+8.7%**
improvement and I nearly spent an afternoon embedding 275,810 library spectra on the strength of
it. At 130 queries every variant is *worse* than plain entropy similarity, and DreaMS alone is far
worse.

**And the mechanism says it should be.** Analog propagation works by *shifting* a neighbour's
spectrum by the neutral-mass difference and matching peaks at the offset. That shift is the entire
idea: two analogs produce the same fragments displaced by a constant. A learned embedding trained
on spectral similarity has no notion of a deliberate m/z offset — to it, a spectrum and the same
spectrum shifted by 42 Da are simply different. DreaMS is answering "are these the same molecule?"
when the channel is asking "is this molecule a *relative* of that one?". The top-10 analogs it
picks are no better by true Tanimoto either (0.2044 vs 0.2038), which confirms it is not finding
better neighbours, it is just finding them no worse.

Two things worth taking from this:

* **A foundation model is not automatically an upgrade for a channel built around a specific
  invariance.** Check whether the pretraining objective contains the invariance your channel
  depends on. Here it does not.
* **n = 40 is not a measurement.** The sign flipped between 40 and 130 queries on the same data.
  If a result is about to justify hours of compute, spend ten minutes tripling the sample first.

Where a pretrained encoder could still pay is the `f·z` channel, which asks the question these
models *are* trained for — but that needs a fingerprint head and GPU fine-tuning, not an
off-the-shelf embedding.


---

## 🔁 Building the cross-encoder, and four ways it can fail

The strongest public notebook at the time of writing ships a component this one does not have. Its
inference wrapper gives the architecture away even though the code is private:

```python
E  = Engine(L, P, tfp, EngineCfg(generate=True), bank, xscorer=xs)
xs = xenc.XScorer(models[0], xencs[0], device=dev)
```

An `XScorer` built from **an fpnet checkpoint *and* a separate `xenc` checkpoint** is a
retrieve-then-rerank stack: a bi-encoder proposes, a cross-encoder re-scores.

The distinction matters here more than usual. This notebook's model channel is a **bi-encoder**:
the spectrum becomes a predicted fingerprint `z`, the candidate is a fingerprint `f`, the score is
`f·z`. That dot product is exactly the CSI:FingerID Bayes log-likelihood, which is why it works —
and it is also the most restrictive possible comparison, because every bit contributes
independently with a fixed weight. The job that holds the remaining score is separating
same-formula isomers, which share most of their bits. That is a question about bit *interactions*,
and a cross-encoder is the natural answer.

So I built one: three interaction terms instead of one (agreement `f·z`, unexpected-present
`f·(1−σ(z))`, expected-missing `(1−f)·σ(z)`), projected and mixed by an MLP, with the raw `f·z`
entering as a residual whose weight starts at 1 and whose MLP starts at zero — so training *begins*
at the bi-encoder score and can only leave it deliberately. Trained on the same 64-way contrastive
task (1 positive, 63 decoys from the same ±10 ppm window).

**It does not work, in four different ways.**

| attempt | training regime | result |
|---|---|---|
| 1 | 276k trunk-seen structures, real `z` | beat `f·z` by +0.005 on its own task; **−0.007** on the holdout |
| 2 | only trunk-held-out structures (1,931), real `z` | train-top1 0.94, **−0.09** on its own validation |
| 3 | 276k structures, `z` corrupted to test-time accuracy | **−0.18**; learned to undo noise that is not there at inference |
| 4 | 276k structures, tiny head, dropout, honest early stopping | delta ≈ **0.000** on validation, **−0.008** on the holdout |

The failures are informative in sequence. Attempt 1 fails because the trunk reaches hardneg-top1
**0.87** on structures it trained on and **0.50** on ones it did not — so the head learns to polish
a good `z` and then meets a poor one. Attempt 2 fixes the regime and runs out of data: holding
structures out of a trunk that saw 276k of them leaves 1,931. Attempt 3 tries to manufacture the
regime with calibrated noise and teaches the head to correct a corruption that never occurs.
Attempt 4 removes enough capacity to stop it hurting, which also removes enough to stop it helping.

**But the real reason is structural, and it is the useful part.** My cross-encoder never sees the
spectrum. Everything it knows arrives through `z`, and `f·z` is already the Bayes-optimal *linear*
readout of `z`. Re-weighting bits can only help if something tells you *which* bits `z` got right
for this particular spectrum — and that information is not in `z`. (Nor is it in the trunk's hidden
state: `z = Wh` with `W` fixed, so `h` carries no more.) A late-interaction head over `(z, f)` is
not a cross-encoder in the sense that matters; it is a re-parameterisation of the same dot product.

A cross-encoder that adds information has to condition on the **peaks**, letting a candidate's
substructures attend to individual fragment signals — which is the same conclusion
[@starkhushi](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/discussion/742304)
reached from the other direction when the MetFrag-lite variants failed to separate isomers:
*separating isomers needs a model of which bonds break, not which masses are reachable.* The
candidate side of that model is a learned fragmenter, not a fingerprint.

That is the honest description of the gap between this notebook and the top of the leaderboard, and
it is an architecture project rather than a configuration change.


---

## 🎯 Which channel actually separates same-formula isomers — and a benchmark that cannot answer it

With pool recall effectively solved (ρ ≈ 1, see the PubChem post-mortem), the score that is left
sits in one place: **choosing between candidates that share the truth's molecular formula.** Mass
cannot separate them by construction — in this holdout the median molecule has **46 same-formula
isomers** inside its ±10 ppm window, and some have 385.

[@starkhushi](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/discussion/742304)
built a bench for exactly this and put the MetFrag-lite channel through it: take the molecules where
the ranker puts the truth at rank 2–3, and ask whether a channel picks the truth out of those three.
Random is 0.333. Six fragmentation variants scored 0.18–0.29; the shipped setting landed at 0.222,
about two standard errors *below* chance.

Running the same bench here reproduces the shape of that result exactly:

| channel | picks the truth out of the top 3 |
|---|---|
| library similarity | 0.000 |
| analog propagation | 0.212 |
| model `f·z` | 0.227 |
| fragmentation | 0.258 |
| *random* | *0.333* |

**Every channel is below chance — and that is the tell.** The cases are selected by the ranker
having failed, and the ranker is built *from these very channels*. Conditioning on "the truth
finished 2nd or 3rd" is conditioning on "the channels preferred the decoys", so each channel is
guaranteed to look sub-random when you measure it there. The bench is measuring its own selection
rule. It cannot distinguish a useless channel from a channel you sampled at its failures.

**The unconfounded question** asks the same thing without conditioning on any ranker: restrict to
the candidates that share the truth's exact mass, and see how each channel ranks the truth inside
that isomer group.

| channel | MRR within the isomer group | vs random |
|---|---|---|
| **analog propagation** | **0.5411** | **+0.372** |
| **model `f·z`** | **0.4904** | **+0.322** |
| fragmentation | 0.2779 | +0.109 |
| library similarity | 0.1175 | −0.051 |
| *random* | *0.1688* | — |

(n = 241, standard error ≈ 0.025.)

Three things fall out of it:

1. **Fragmentation does separate isomers** — +0.109 over random, more than four standard errors.
   The opposite of what the conditioned bench suggested. It is the weakest of the three real
   channels, not a dead one, and that is consistent with its measured +0.021 on the leaderboard
   when it was added.
2. **Analog propagation is the strongest isomer discriminator**, slightly ahead of the model. Both
   are roughly three times random. This is where the remaining score lives.
3. **Library similarity is actively anti-informative here** (−0.051, about two standard errors
   below random). That is not a bug: a Class-2 molecule has no reference spectrum, so any library
   signal attached to one of its isomers is evidence for the *wrong* molecule. It is worth knowing
   that the channel carrying 45% of the total score is a liability on the half of the problem that
   still has headroom.

**The general point is worth more than the numbers.** If you evaluate a component on cases selected
by a system that already contains that component, you will measure the selection, not the
component. Pick the evaluation set with something independent of what you are testing — here, the
molecular formula, which is a fact about chemistry rather than an opinion of the ranker.


---

## 🎲 Deciding a bet your validation set cannot score

Pool expansion is the one lever this notebook's holdout is **structurally unable to evaluate**.
Every holdout answer is already in the pool — local recall is **100%** at ±10 ppm — so adding a
database can only ever show up as dilution, never as the recall it buys. Running the experiment and
reading off "it got worse" is not a measurement. It is the shape of the instrument.

So the decision gets made with algebra, and each term gets measured separately.

### The inequality

Let $\rho$ be the pool's true recall on hidden Class-2 molecules, $r(K)$ the chance a
PubChem-supplied answer survives an admission cut of $K$, and $d(K)$ the dilution factor. Expansion
pays exactly when

$$\underbrace{\big[\rho + (1-\rho)\,r(K)\big]}_{\rho'} \cdot d(K) \;>\; \rho
\qquad\Longleftrightarrow\qquad
\rho \;<\; \rho^{*} = \frac{r d}{1 - d + r d}$$

Three of these are measurable without knowing $\rho$.

### Measuring r(K): does the model find the answer among 71 million isomers?

Take a holdout query, **pretend its answer is not in the pool**, pull the PubChem isomers inside the
same ±10 ppm window, and ask where the model's $f\cdot z$ ranks the true structure.

The windows are large — a median of several thousand isomers, up to 30,000. The answer is present in
**72%** of them (stereoisomers collapse onto the same skeleton, so it survives subsampling), and the
model ranks it like this:

| admitted $K$ | 5 | 10 | 20 | 35 | 50 |
|---|---|---|---|---|---|
| $r(K)$ | 0.268 | 0.360 | 0.464 | 0.540 | **0.572** |

Better than it sounds: rank **#1 out of ~1,800 isomers** happened repeatedly. This is the `f·z`
channel doing exactly what CSI:FingerID promises.

### Measuring d(K): what the extra candidates cost

Same holdout, answer left in the pool, so this is pure dilution:

| admitted $K$ | 5 | 10 | 20 | 35 | 50 |
|---|---|---|---|---|---|
| Class-2 $d(K)$ | 0.740 | 0.711 | 0.685 | 0.656 | **0.650** |
| break-even $\rho^{*}$ | 0.433 | 0.470 | 0.503 | 0.507 | **0.515** |

My intuition said a small $K$ would be safer — fewer intruders, less damage. **The arithmetic says
the opposite**: $r$ falls faster than $d$ recovers, so $\rho^{*}$ *rises* with $K$ over this range.
Admitting all ~1,800 isomers is worse again ($d \approx 0.52$), so the optimum is a broad plateau
around $K = 35$–$50$.

**Class 1 is barely touched**: $d_{c_1} = 0.941$ (0.901 → 0.848). A Class-1 molecule has real
library evidence, and the ranker's library features dominate, so PubChem intruders cannot displace
it. The damage is confined to the class with nothing to defend it — the same asymmetry as the
candidate cap, this time with a much gentler slope.

### Two refinements, one of which failed

**Failed: normalising the admission score.** `f·z` is a *sum over set bits*, so a candidate with a
larger fingerprint scores higher for free — the cut might be selecting big molecules rather than
likely ones. Dividing by the bit count (or its square root) should fix that. It does not:

| admission rule | retain@20 | retain@35 | retain@50 | retain@80 |
|---|---|---|---|---|
| **raw `f·z`** | **0.464** | **0.540** | **0.572** | **0.632** |
| `f·z / √bits` | 0.424 | 0.516 | 0.544 | 0.608 |
| `f·z / bits` | 0.364 | 0.460 | 0.532 | 0.592 |

Raw wins at every cut. The bit count is not a nuisance term — a bigger molecule genuinely has more
predicted substructure to match, and `f·z` is the Bayes log-likelihood precisely as written. (Note
that z-scoring within the candidate set is a *monotone* transform and cannot change a ranking, so
bit-count normalisation is the only knob here.)

**Worth doing: widen the window before you sharpen the cut.** `retain(50) = 0.572` factors as

$$\underbrace{0.788}_{\text{answer is in the sampled window}} \times \underbrace{0.79}_{\text{and survives the top-50 cut}}$$

The cut is not the bottleneck — **presence** is, and presence is set purely by how many isomers you
can afford to fingerprint. The windows are large (median 4,772 isomers, p90 16,291, max 36,518):

| isomers fingerprinted per query | 2,000 | 5,000 | 12,000 | all |
|---|---|---|---|---|
| presence | 0.788 | 0.884 | 0.940 | 0.948 |

Raising the cap to 12,000 would lift `retain(50)` to ≈0.74 and the break-even to $\rho^{*} \approx
0.58$, at roughly 6× the fingerprinting cost (about 3 hours of notebook runtime). That is the
cheapest remaining improvement to this lever, and it is a pure engineering budget question rather
than a modelling one.

### The verdict: I took the bet, and it lost — decisively

**0.335 → 0.205.**

That is worse than my *worst* modelled case ($\rho = 0.99$ predicted 0.261), and the gap is
informative. Working backwards with the measured Class-1 dilution:

$$0.205 = \underbrace{0.162 \times 0.93 \times 0.941}_{0.142,\ \text{Class 1}} + 0.184 \times m
\quad\Longrightarrow\quad m = 0.344$$

But $m = (\rho'/\rho) \cdot d_{c_2}$ and $\rho' \ge \rho$ always, so $m \ge d_{c_2}$. My local
$d_{c_2} = 0.650$ cannot produce $m = 0.344$. **The real dilution is roughly twice as damaging as
the holdout says**, and the recall gain is nearly zero.

Both halves of that are worth understanding.

**Why dilution is worse in the wild.** My holdout's Class-2 MRR is 0.74; the leaderboard-implied
value is ~0.68, and more to the point the *margin* is thinner — the ranker is less sure about hidden
Class-2 molecules than about mine. Intruders displace an answer the ranker barely preferred far more
easily than one it preferred confidently. Measuring dilution on queries you rank well systematically
understates it.

**Why the recall gain is nearly zero — and what that settles.** Expansion only loses this badly if
$\rho$ is already high: the pool *does* contain the Class-2 answers. So the bracket collapses to the
top of its range, $\rho \approx 0.9\text{–}1.0$, and the hosts' 250-molecule example set (99.6%
COCONUT coverage) was the better guide all along. The pooled 37.7% figure was dragged down by GNPS,
which is not what this test set looks like.

Two consequences, and they matter more than the submission did:

1. **$f_2 = B/\rho = 0.271/\rho \approx 0.27\text{–}0.30$.** The Class-2 share sits essentially *at*
   its hard lower bound. Class 3 is therefore around 55% of the test — the majority of this
   competition is molecules no retrieval system can reach at all.
2. **Recall is not the bottleneck. Ranking is.** Earlier versions of this notebook said the
   opposite, confidently, and were wrong. With $\rho \approx 1$ the ceiling $A + B \approx 0.43$ is
   fully available to a better ranker, and every point between here and there is ordering, not
   coverage. Do not spend your time on bigger databases.

**And the meta-lesson, which is why this section stays.** A 0.130 loss is a large price for one
number, but the number was otherwise unobtainable: no amount of local work could have told me
$\rho$, because my instrument was built in a way that guaranteed $\rho = 1$ locally. When a quantity
is structurally invisible to your validation, a submission is not a gamble — it is the measurement.
Budget for it, take the bounded downside, and write down what it tells you.



---

## 📐 How big is Class 2 really? (an open question, sharpened)

A reader worked through my numbers and got a different answer, which is worth addressing head-on
because the disagreement is real and instructive.

What the leaderboard actually pins down is a **product**, not two separate numbers:

$$\text{Class-2 contribution} \;=\; \underbrace{f_2}_{\text{share of test}} \times \underbrace{\rho}_{\text{pool recall}} \times \underbrace{c_2}_{\text{ranking quality}}$$

From the submissions: Class-1 contributes $0.162 \times 0.87 \approx 0.141$, so at LB **0.266** the
Class-2 contribution is $\approx 0.125$, and with $c_2 = 0.539$ that gives

$$f_2 \times \rho \;\approx\; 0.233$$

**That is all the leaderboard tells you.** Splitting it needs an assumption about $\rho$:

* Assume $\rho \approx 0.996$ (COCONUT's coverage of `enveda-np-examples`) → $f_2 \approx 23\%$.
* Assume $\rho \approx 0.5$ → $f_2 \approx 47\%$.

I think the first assumption is optimistic, for a specific reason: **99.6% is measured on compounds
that already have public reference spectra.** A Class-2 molecule is *defined* by having none — it is
exactly the kind of compound a curated natural-product database is least likely to contain. Using
in-library compounds to estimate coverage of out-of-library compounds is the same selection bias
that made my own holdout leak (see the ranker section).

There is also independent evidence from the leaderboard itself. A team at **0.332** with the same
saturated Class-1 ceiling has a Class-2 contribution of $\approx 0.19$. Even granting them a
generous $c_2 = 0.6$ and perfect recall, that needs $f_2 \gtrsim 0.32$; at a realistic $c_2 \approx
0.55$ it needs $f_2 \times \rho \approx 0.35$ — **half again my 0.233**. Since ranking quality
cannot plausibly differ by that much, the likeliest explanation is that **their $\rho$ is higher
than mine**, i.e. a broader candidate database.

So my working estimate is $f_2 \approx 35\text{–}50\%$ with $\rho \approx 0.5\text{–}0.65$ — but I
want to be clear this is an *inference from the leaderboard*, not a measurement.

### Update: a hard lower bound, which settles part of the argument

As the ranker improved, the fitted product rose from $0.233 \to 0.271$. That gives a bound needing
**no assumption about $\rho$ at all**, because $\rho$ is a probability:

$$f_2 \;=\; \frac{f_2\rho}{\rho} \;\ge\; f_2\rho \;=\; 0.271 \qquad (\rho \le 1)$$

**So $f_2 \ge 27\%$, and $f_3 \le 1 - 0.162 - 0.271 \approx 57\%$.** An estimate of $f_2 \approx 18\%$
is not merely unlikely, it is inconsistent with the leaderboard: a test set with 18% Class 2 cannot
produce a Class-2 contribution of 0.271 even with a perfect pool and a perfect ranker.

The bound is also **conservative in the safe direction**. $f_2\rho$ was obtained by dividing the
measured Class-2 contribution by *my* $c_2$, and my holdout is almost certainly easier than the
hidden test's Class-2 slice (see the section on validation not transferring). If the true $c_2$ is
lower than I measure, then $f_2\rho$ is *larger* than 0.271, and the bound only rises.

### The same algebra turns "should I add PubChem?" into one inequality

Adding a database raises recall $\rho \to \rho'$ and dilutes ranking by a factor
$d = c_2^{\text{new}}/c_2$. It is worth doing exactly when

$$\rho' d > \rho.$$

PubChem's dilution is measured here at $d \approx 0.52$ (Class-2 MRR $0.73 \to 0.38$ with the model
channel), and $\rho' \approx 1$ by construction if the hosts define Class 2 by PubChem membership.
So **PubChem pays iff $\rho < 0.52$, i.e. iff $f_2 > 0.271/0.52 \approx 52\%$.**

That is a genuinely close call, and it sits right at the top of my estimated range.

**And here is the uncomfortable part: this notebook cannot measure it.** Every one of my holdout
answers is already in the pool — local recall is **100%** at plus/minus 10 ppm. So a pool expansion
can only ever show me its dilution and never its recall gain. When I reported "PubChem hurts,
measured, do not", that was an honest measurement of *one side of the ledger*, presented as though
it were both. It should read: **PubChem costs 48% of ranking quality, and buys an amount of recall
I have no local instrument to see.**

### Measuring $\rho$: what COCONUT actually covers

So I went and measured it, as far as it can be measured. A Class-2 molecule has no public reference
spectrum, so it cannot be sampled from a spectral library — but the structures *in* the training
libraries come from the same chemistry, and asking how many of them COCONUT contains brackets the
answer:

| library | structures | in COCONUT |
|---|---|---|
| `enveda-np-examples` (the hosts' own example set) | 250 | **99.6%** |
| masaryk | 416 | 83.7% |
| mona | 11,681 | 79.4% |
| msdial | 9,127 | 78.4% |
| riken | 15,892 | 77.2% |
| drug_plus | 2,539 | 75.3% |
| massbank | 9,180 | 61.4% |
| gnps | 45,750 | **36.4%** |
| spectraverse | 9,631 | 21.6% |
| **all NP libraries pooled** | **64,941** | **37.7%** |

Two things jump out.

**The spread is enormous** — 22% to 99.6%. COCONUT's coverage is not a property of natural products,
it is a property of *how curated a particular collection is*. The pooled figure is dragged to 37.7%
by GNPS, which is the largest and least curated of them.

**The 99.6% figure comes from a 250-molecule curated example set**, and every one of those molecules
has public reference spectra — which is exactly the population a natural-product database is *most*
likely to already contain. It is the best-case end of the range, not the typical case. Every number
in that table is biased the same way, upward, because all of them have public spectra and Class-2
molecules by definition do not. So **37.7% is an upper bound on a pooled estimate that is itself an
upper bound.**

Put that next to the decision rule: PubChem pays iff $\rho < 0.52$. The curated end of the range says
no; the pooled end says yes. The threshold sits *inside* the bracket, so it cannot be settled from
these numbers — which is exactly why it was worth a submission.

### Answered — by submission, at a cost of 0.130

I ran the expansion and it scored **0.205** against the 0.335 it replaced. That is worse than even
the $\rho = 0.99$ case predicted, so the recall gain was ~zero: **$\rho$ is at the top of the
bracket**, and the hosts' own example set was the better guide. The pooled 37.7% was dragged down by
GNPS, which is not what this test set looks like. Therefore

$$f_2 \;=\; \frac{B}{\rho} \;=\; \frac{0.271}{\approx 1} \;\approx\; \mathbf{0.27\text{–}0.30},
\qquad f_3 \;\approx\; \mathbf{0.55}$$

The Class-2 share sits essentially *at* the hard lower bound derived above, and **the majority of
this test set is Class 3** — molecules no retrieval method can reach at all. That is why the
leaderboard compresses so hard just above 0.35: everyone is converging on the same reachable
fraction.

To the reader who put $f_2$ nearer 18%: the bound $f_2 \ge 0.271$ still rules that out, but your
instinct that my ~45% was too high was right, and for the reason you questioned — I treated $\rho$
as low when it is high. The disagreement is what sent me to measure it, so thank you for it.

**Where this says to spend effort:** pool coverage is a solved problem in this competition. With
$\rho \approx 1$ the ceiling $A + B \approx 0.43$ is fully available, and everything between here and
there is *ordering*, not coverage. Earlier versions of this notebook said the opposite — that recall
was the bottleneck — and said it confidently. That was wrong, and this is the measurement that
corrects it.


---

## What I measured, and what I'd try next

**Things that worked**

* Mass-shifted analog propagation — the single biggest classical lever (0.16 → 0.52 on Class 2).
* A spectrum→fingerprint model ranked by `f·z`, trained on **fused multi-spectrum inputs** —
  **0.517 alone**, level with analog propagation, and **0.63 with all four channels**. Worth +0.033
  on the leaderboard (0.266 → 0.299) the first time it went in, and another +0.036 when it was
  retrained longer.
* Sweeping the ranker's hyperparameters against the leaderboard-calibrated objective rather than
  against validation MRR (+0.007 predicted), and **re-sweeping them whenever a channel improves** —
  the class-prior optimum moved from ~0.50 to 0.45 once the model channel went 0.490 → 0.517, and
  the wide hedge that used to help turned into a 0.008 liability.
* Entropy similarity with entropy weighting instead of plain cosine (Class 1: 0.893 → 0.93).
* A **tight** ±10 ppm candidate window. Tighter is genuinely better — but only at the *window*
  stage, where the filter is mass and mass is what a window is for. Do **not** extend the same
  intuition to a post-window cap (see the candidate-cap section: it costs 24% of Class-2 recall).
* Calibrating the class weight against leaderboard-derived values rather than the raw class share.
* **Bagging the ranker over fixed seeds.** `HistGradientBoostingClassifier` defaults to
  `random_state=None`; two byte-identical notebook versions scored **0.292 and 0.298**. See the
  noise-floor section — this is the single most common way to fool yourself in this competition.

**Measured dead ends — save yourself the time**

| Idea | Result |
|---|---|
| Add **all** PubChem isomers to the pool | 0.52 → 0.35. But this measures only PubChem's *dilution* — local pool recall is already 100%, so the instrument is blind to the recall it buys. Admitting the model's **top 50** instead of all of them changes the arithmetic completely: see *Deciding a bet your validation set cannot score* |
| Soft "consensus fingerprint" averaged over analogs | 0.43 vs 0.52 for max-over-analogs |
| Per-instrument normalisation of analog similarity | 0.517 vs 0.521 |
| Confidence gate to detect "answer not in pool" | AUC 0.627 — too weak to route on |
| Averaging more models **within one input view** | a second merged-input model took the leaderboard from 0.335 → **0.329**. Two models help only when they read the spectra *differently* (per-spectrum vs merged); a second reading of the same kind is redundant |
| Averaging **one** model over two fusion views (per-spectrum *and* merged peak list) | merged-only **0.5163**, per-spectrum 0.4946, mean 0.5087, best weighted blend 0.5134. Even with the scale shared, test-time augmentation loses: feed the model the input distribution it was *trained* on and stop there |
| Re-weighting analogs by how well they fit the predicted fingerprint | 0.521 → 0.512 |
| ChEBI + LIPID MAPS expansion | looked net-positive on validation, leaderboard "disagreed" (0.299 → 0.295) — but that gap is **inside the seed-noise band**, so this one is genuinely *unresolved*, not refuted |
| 3 reference spectra per structure instead of 1 | 0.52 → 0.49 |
| Wider analog window (±400 Da vs ±200) | no change (0.520 vs 0.521) |
| NP-likeness as a prior | 0.037 — worse than random |
| Explicit molecular-formula prediction | only a 1.4× candidate reduction; 72% of window candidates already share the true formula |
| Library similarity as a fixed additive term | Class 2 0.52 → 0.27 |
| Coarse candidate cap at 80 by `lib_sim×100 − \|Δmass\|` (as several forks do) | **−0.055 predicted LB**; drops 24% of Class-2 truths outright. Class-1 retention is 0.992 at every cap, which is why it looks harmless |
| Simple z-score blend of the four channels instead of the GBM | 0.606 vs 0.620 |
| Test-time augmentation: averaging one model over per-spectrum **and** merged views | merged-only 0.516 beats the mean 0.509 and every weighted blend |
| **Mass error as a soft ranker feature** | The truth's mass-error percentile inside the window has median **0.017** and lands in the best 10% for **68%** of queries — because 72% of candidates share its exact formula and tie at the minimum, so this is really a "right molecular formula" signal. Real information, and still a **wash**: query-held-out CV gives c₁ 0.818 → 0.855 and c₂ 0.627 → 0.605, which cancel exactly (0.3024 → 0.3025). It helps the class that does not need help |
| Giving the ranker **both model views separately** instead of their average, plus agreement features | 0.3042 → 0.3024. Pairing the two views is worth +0.019 on the leaderboard, so their *disagreement* looked like exploitable signal. It is not — the ranker cannot use it, and the plain average is better |
| Targeted derivative enumeration (±O, ±CH₂, ±hexose, +acetyl, ±2H on strong analogs) | 0.337 → 0.335, measured by [@starkhushi](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/discussion/742055). This notebook proposed it as the most promising untried pool idea; it is not. Another point for "recall is not the bottleneck" |
| More analogs per molecule | flat above 80: `N_ANALOG` 60 → 0.3683, 80 → 0.3709, 100 → 0.3705, 140 → 0.3703 predicted LB. Nothing here; spend the effort elsewhere |

**Where the remaining score is — and the honest ceiling**

Submissions pin the unknowns in `ExpectedLB = A·c₁ + B·c₂`, where `A = f₁` and `B = f₂ · recall`:

* A library-only submission scores `0.151`, and a deliberately hard cross-library Class-1
  simulation (query with one source library, search with that library **removed**, n=500) gives
  c₁ = 0.872–0.93, with six similarity variants landing within 0.01 of each other. So
  **A = f₁ ≈ 0.162**, and 0.151 is already Class 1's *entire* value. That channel is saturated.
* `B` is then whatever reconciles the equation with the leaderboard. Fitted repeatedly, it has
  drifted **0.20 → 0.233 → 0.271** as the ranker improved. That drift is informative, and it is not
  a constant of nature: it means my Class-2 validation split is systematically *harder* than the
  hidden test's Class-2 slice, so every real improvement lands **larger** on the leaderboard than
  validation predicts. Re-fit `B` after each scored submission; don't treat it as measured.

At the current `A = 0.162`, `B = 0.271`, a leaderboard score of `0.335` implies a hidden-test
c₂ ≈ **0.68** — well above the 0.62 this notebook measures on its own Class-2 holdout, exactly as
the drift predicts.

The ceiling this pool allows is `A + B = 0.162 + 0.271 ≈ **0.43**` — that is what a *perfect*
ranker would score, because `B` already folds in how often the answer is in the pool at all.
So there is ≈0.10 of ranking headroom left and an unknown amount of recall headroom above it.

**What I'd try next, in order of expected value**

Two submissions rewrote this list, so read it in that order.

1. **A second *input view*, not a second model.** The largest effect I ever isolated on this
   leaderboard is **+0.019** for pairing a per-spectrum model with a merged-input one. Adding a
   *third* model of the same kind gave **−0.006**. So the win is representational: two genuinely
   different readings of the same spectra. A third view — different peak preprocessing, a
   collision-energy-conditioned encoder, an MS1-aware branch — is the highest-value untried idea
   here, and it is cheap compared with training a bigger net.
2. **A better per-spectrum model.** The pair is only as good as its weaker half, and mine is a
   20,000-step model. That is the single most under-trained component in the stack.
3. **Ranking, not recall.** This used to say the opposite. The PubChem submission settled it:
   $\rho \approx 1$, the pool already holds the Class-2 answers, and the full ceiling
   $A + B \approx 0.43$ is available to a better ranker. **Do not spend time on bigger databases.**
   With $f_3 \approx 0.55$ unreachable by construction, the competition is a fight over ordering
   inside a pool everyone already has.
4. **More ranker seeds.** Free variance reduction against a 0.0072 noise floor — and nobody should
   be reading 0.004 deltas as signal. But tune with folds held out **by query**; see the ranker
   section for the 0.005 that mistake cost me.
5. **Better Class-1 recall** for molecules whose only reference spectra come from a different
   instrument family. Class 1 is saturated at ~0.93 within this approach, but it is 45% of the score.

Ideas, corrections and forks very welcome — the constants are all in `CFG`. 🙂
